# Phase 3 · Rhetorical role prediction

Classify each sentence in a judgment using its text and nearby sentences from the same document. The four Phase 1 backbones are evaluated with full fine-tuning, LoRA, LoRMA, and IA³. Knowledge distillation uses a completed direct experiment as the teacher.

The notebook saves checkpoints, development scores, and test submissions under `/kaggle/working/phase3_rr_v2`.

## Required files and environment

- Attach `phase3_model_assets/` from the Phase 3 assets ZIP. Each of the four model folders contains `best.pt`, `tokenizer.json`, and `run_config.json`.
- Attach the official `Phase_3_RR/` data and `evaluate_phase3.py`. To download the data from the course repository in Kaggle, enable Internet and set the `HF_TOKEN` secret. The saved run used revision `89edba6a44d6bb9593fd13b6895b5cd3b06eb82e`.
- Enable a GPU. Use the path overrides in the settings cell if more than one attached copy is found.

In [1]:
import subprocess, sys
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "numpy>=1.26,<3", "scikit-learn>=1.4,<2", "huggingface-hub>=0.30,<2",
    "matplotlib>=3.8,<4"
], check=True)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'numpy>=1.26,<3', 'scikit-learn>=1.4,<2', 'huggingface-hub>=0.30,<2', 'matplotlib>=3.8,<4'], returncode=0)

In [2]:
import copy
import gc
import hashlib
import heapq
import itertools
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import time
import zipfile
from collections import Counter, defaultdict
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import HTML, display
import base64
import html
from sklearn.metrics import f1_score, precision_score, recall_score
from torch.utils.checkpoint import checkpoint
from torch.utils.data import DataLoader, Dataset

PAD, UNK, BOS, EOS = range(4)

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary.replace(path)

def digest_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def normalize_id(value):
    value = str(value)
    return value[:-4] if value.endswith(".txt") else value

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available(), "GPUs:", torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print(index, props.name, round(props.total_memory / 2**30, 2), "GiB")


PyTorch: 2.11.0+cu128
CUDA: True GPUs: 2
0 Tesla T4 14.56 GiB
1 Tesla T4 14.56 GiB


In [3]:
INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/phase3_rr_v2")
ASSET_DIR = OUTPUT_ROOT / "assets"

MODEL_DIRS = {
    "model1": None,
    "model2": None,
    "model3": None,
    "model4": None,
}
DATA_ROOT = None
EVALUATOR_PATH = None

REPO = "Exploration-Lab/CS-781-Capstone"
DATA_REVISION = None

MODEL_NAMES = ("model1", "model2", "model3", "model4")
DIRECT_METHODS = ("full", "lora", "lorma", "ia3")

CONFIG = {
    "seed": 42,
    "max_length": 512,
    "left_context_sentences": 2,
    "current_sentence_fraction": 0.75,
    "right_context_sentences": 1,
    "current_truncation": "head_tail",
    "epochs": 12,
    "early_stopping_patience": 3,
    "effective_batch": 32,
    "micro_batch_per_gpu": 2,
    "activation_checkpointing": True,
    "full_lr": 2e-5,
    "peft_lr": 1e-4,
    "head_lr": 3e-4,
    "min_lr_ratio": 0.1,
    "warmup_fraction": 0.06,
    "weight_decay": 0.01,
    "clip": 1.0,
    "class_weight_power": 0.5,
    "class_weight_cap": 5.0,
    "dropout": 0.2,
    "label_smoothing": 0.0,
    "inference_batch_size": 8,
    "num_workers": 2,
    "capacity_headroom_gib": 2.0,
    "max_skipped_updates": 30,
    "rank": 16,
    "alpha": 16.0,
    "log_every": 20,
    "save_every": 100,
    "max_hours_per_call": 10.5,
    "save_optimizer": True,
    "kd_temperature": 2.0,
    "kd_weight": 0.5,
}

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ASSET_DIR.mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, indent=2))


{
  "seed": 42,
  "max_length": 512,
  "left_context_sentences": 2,
  "current_sentence_fraction": 0.75,
  "right_context_sentences": 1,
  "current_truncation": "head_tail",
  "epochs": 12,
  "early_stopping_patience": 3,
  "effective_batch": 32,
  "micro_batch_per_gpu": 2,
  "activation_checkpointing": true,
  "full_lr": 2e-05,
  "peft_lr": 0.0001,
  "head_lr": 0.0003,
  "min_lr_ratio": 0.1,
  "warmup_fraction": 0.06,
  "weight_decay": 0.01,
  "clip": 1.0,
  "class_weight_power": 0.5,
  "class_weight_cap": 5.0,
  "dropout": 0.2,
  "label_smoothing": 0.0,
  "inference_batch_size": 8,
  "num_workers": 2,
  "capacity_headroom_gib": 2.0,
  "max_skipped_updates": 30,
  "rank": 16,
  "alpha": 16.0,
  "log_every": 20,
  "save_every": 100,
  "max_hours_per_call": 10.5,
  "save_optimizer": true,
  "kd_temperature": 2.0,
  "kd_weight": 0.5
}


In [4]:
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    for secret_name in ("HF_TOKEN", "HF"):
        try:
            token = secrets.get_secret(secret_name)
            if token:
                os.environ["HF_TOKEN"] = token
                break
        except Exception:
            pass
except Exception:
    pass


In [5]:
def classify_model_config(config):
    if "encoder_layers" in config and "decoder_layers" in config:
        return "model2"
    if "kernel_size" in config and "dilation_cycle" in config:
        return "model4"
    if "window" in config and "global_every" in config:
        return "model3"
    if "layers" in config and "heads" in config and "context" in config:
        return "model1"
    return None

def checkpoint_metadata(path):
    state = torch.load(path, map_location="cpu", weights_only=False)
    config = state.get("model_config")
    if not isinstance(config, dict):
        raise ValueError(f"No model_config in {path}")
    return classify_model_config(config), config

def discover_model_dirs():
    candidates = defaultdict(list)
    for config_path in INPUT_ROOT.rglob("run_config.json"):
        folder = config_path.parent
        if not (folder / "best.pt").is_file() or not (folder / "tokenizer.json").is_file():
            continue
        try:
            payload = read_json(config_path)
            kind = classify_model_config(payload.get("model", {}))
            if kind:
                candidates[kind].append(folder)
        except (OSError, ValueError, json.JSONDecodeError):
            pass
    for kind in MODEL_NAMES:
        if MODEL_DIRS[kind] is None:
            unique = sorted(set(candidates[kind]))
            if len(unique) != 1:
                raise ValueError(f"Set MODEL_DIRS['{kind}'] explicitly. Candidates: {unique}")
            MODEL_DIRS[kind] = unique[0]
        MODEL_DIRS[kind] = Path(MODEL_DIRS[kind])

def has_phase3_files(root):
    required = [
        "Train/CL_train.json", "Train/CL_dev.json", "Train/IT_train.json",
        "Train/IT_dev.json", "Train/label_vocab.json",
        "Test/CL_test.json", "Test/IT_test.json",
    ]
    return all((root / name).is_file() for name in required)

def discover_data_root(explicit=None):
    if explicit is not None:
        root = Path(explicit)
        if not has_phase3_files(root):
            raise FileNotFoundError(f"Incomplete Phase-3 folder: {root}")
        return root
    found = []
    for marker in INPUT_ROOT.rglob("label_vocab.json"):
        root = marker.parent.parent
        if has_phase3_files(root):
            found.append(root)
    found = sorted(set(found))
    if len(found) == 1:
        return found[0]
    if len(found) > 1:
        raise ValueError(f"Set DATA_ROOT explicitly. Candidates: {found}")
    from huggingface_hub import snapshot_download
    kwargs = dict(repo_id=REPO, repo_type="dataset", allow_patterns=["Phase_3_RR/**", "**/Phase_3_RR/**"])
    if DATA_REVISION is not None:
        kwargs["revision"] = DATA_REVISION
    snapshot = Path(snapshot_download(**kwargs))
    found = [p.parent.parent for p in snapshot.rglob("label_vocab.json") if has_phase3_files(p.parent.parent)]
    found = sorted(set(found))
    if len(found) != 1:
        raise FileNotFoundError(f"Could not resolve Phase_3_RR under {snapshot}: {found}")
    return found[0]

discover_model_dirs()
DATA_ROOT = discover_data_root(DATA_ROOT)
if EVALUATOR_PATH is None:
    evaluator_candidates = sorted(INPUT_ROOT.rglob("evaluate_phase3.py"))
    if len(evaluator_candidates) != 1:
        raise ValueError(f"Set EVALUATOR_PATH explicitly. Candidates: {evaluator_candidates}")
    EVALUATOR_PATH = evaluator_candidates[0]
EVALUATOR_PATH = Path(EVALUATOR_PATH)

for name, folder in MODEL_DIRS.items():
    print(name, folder)
print("Data:", DATA_ROOT)
print("Evaluator:", EVALUATOR_PATH)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

model1 /kaggle/input/datasets/acadsiitk/phase3-req/phase3_model_assets/model1_decoder
model2 /kaggle/input/datasets/acadsiitk/phase3-req/phase3_model_assets/model2_encoder_decoder
model3 /kaggle/input/datasets/acadsiitk/phase3-req/phase3_model_assets/model3_local_global
model4 /kaggle/input/datasets/acadsiitk/phase3-req/phase3_model_assets/model4_gated_cnn
Data: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_3_RR
Evaluator: /kaggle/input/datasets/acadsiitk/evaluator-phase-3/evaluate_phase3.py


## Tokenizer and Phase 1 models


In [6]:
def split_bpe_pieces(text, max_chars=256):
    """Keep every character; separate words, punctuation and whitespace.

    An optional leading ASCII space can join a word or punctuation piece.
    Long pieces are split into bounded chunks to limit per-piece work.
    No normalization, strip(), split(), external tokenizer or regex package.
    """
    for match in re.finditer(' ?\\w+| ?[^\\w\\s]+|\\s+', text):
        piece = match.group(0)
        for start in range(0, len(piece), max_chars):
            yield piece[start:start + max_chars].encode('utf-8')


def apply_bpe_merges(piece, merge_ranks):
    """Apply learned merges by rank, breaking occurrence ties left-to-right.

    Token IDs 0..3 are reserved. Byte b always has ID b+4. A linked list
    and a heap avoid repeatedly scanning every adjacent pair in a piece.
    """
    ids = [byte + 4 for byte in piece]
    size = len(ids)
    if size < 2 or not merge_ranks:
        return tuple(ids)
    previous = [i - 1 for i in range(size)]
    following = [i + 1 for i in range(size)]
    following[-1] = -1
    active = [True] * size
    queue = []

    def offer(left):
        if left < 0 or not active[left]:
            return
        right = following[left]
        if right < 0:
            return
        pair = (ids[left], ids[right])
        rank = merge_ranks.get(pair)
        if rank is not None:
            heapq.heappush(queue, (rank, left, right, pair[0], pair[1]))
    for left in range(size - 1):
        offer(left)
    while queue:
        rank, left, right, a, b = heapq.heappop(queue)
        if not active[left] or not active[right] or following[left] != right:
            continue
        if ids[left] != a or ids[right] != b:
            continue
        ids[left] = 260 + rank
        active[right] = False
        following[left] = following[right]
        if following[right] >= 0:
            previous[following[right]] = left
        offer(previous[left])
        offer(left)
    result = []
    position = 0
    while position >= 0:
        result.append(ids[position])
        position = following[position]
    return tuple(result)


class ScratchByteBPE:
    """Our byte BPE vocabulary, encoder, decoder and JSON serialization.

    All base bytes and EVERY intermediate merge token are retained.
    Integer IDs identify symbols; concatenated decimal strings are never used.
    Literal '<s>' etc. in ordinary input are encoded as text, not control IDs.
    """
    FORMAT = 'handwritten-byte-bpe-v1'
    PATTERN = ' ?\\w+| ?[^\\w\\s]+|\\s+'

    def __init__(self, merges=None, max_piece_chars=256):
        self.specials = ['<pad>', '<unk>', '<s>', '</s>']
        self.max_piece_chars = max_piece_chars
        if not isinstance(max_piece_chars, int) or max_piece_chars < 1:
            raise ValueError('max_piece_chars must be a positive integer')
        self.merges = []
        self.merge_ranks = {}
        self.token_bytes = [None] * 4 + [bytes([b]) for b in range(256)]
        self._cache = {}
        for pair in merges or []:
            self.add_merge(tuple(pair))

    def add_merge(self, pair):
        if len(pair) != 2 or pair in self.merge_ranks:
            raise ValueError('Invalid or repeated merge pair')
        if any((type(token) is not int or not 4 <= token < len(self.token_bytes) for token in pair)):
            raise ValueError('A merge must reference earlier non-special token IDs')
        new_id = len(self.token_bytes)
        self.merge_ranks[pair] = len(self.merges)
        self.merges.append(pair)
        self.token_bytes.append(self.token_bytes[pair[0]] + self.token_bytes[pair[1]])
        self._cache.clear()
        return new_id

    def get_vocab_size(self):
        return len(self.token_bytes)

    def token_to_id(self, special):
        return self.specials.index(special) if special in self.specials else None

    def encode(self, text):
        if not isinstance(text, str):
            raise TypeError('encode expects a string')
        output = []
        for piece in split_bpe_pieces(text, self.max_piece_chars):
            encoded = self._cache.get(piece)
            if encoded is None:
                encoded = apply_bpe_merges(piece, self.merge_ranks)
                if len(self._cache) >= 65536:
                    self._cache.clear()
                self._cache[piece] = encoded
            output.extend(encoded)
        return SimpleNamespace(ids=output)

    def decode(self, ids, skip_special_tokens=True):
        chunks = []
        for value in ids:
            token = int(value)
            if not 0 <= token < len(self.token_bytes):
                raise ValueError(f'Unknown token ID: {token}')
            if token < 4:
                if not skip_special_tokens:
                    chunks.append(self.specials[token].encode('utf-8'))
            else:
                chunks.append(self.token_bytes[token])
        return b''.join(chunks).decode('utf-8', errors='replace')

    def save(self, path, training=None):
        payload = {'format': self.FORMAT, 'special_tokens': self.specials, 'byte_offset': 4, 'pattern': self.PATTERN, 'max_piece_chars': self.max_piece_chars, 'merges': self.merges, 'vocab_hex': [None if value is None else value.hex() for value in self.token_bytes]}
        if training is not None:
            payload['training'] = training
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_suffix(path.suffix + '.tmp')
        temporary.write_text(json.dumps(payload, ensure_ascii=False), encoding='utf-8')
        temporary.replace(path)

    @classmethod
    def from_file(cls, path):
        payload = json.loads(Path(path).read_text(encoding='utf-8'))
        if payload.get('format') != cls.FORMAT:
            raise ValueError('This is not our handwritten BPE format. Use a fresh data directory and retrain the tokenizer/token arrays.')
        if payload.get('special_tokens') != ['<pad>', '<unk>', '<s>', '</s>']:
            raise ValueError('Special-token IDs do not match')
        if payload.get('byte_offset') != 4 or payload.get('pattern') != cls.PATTERN:
            raise ValueError('Byte mapping or splitting rules do not match')
        tokenizer = cls(payload['merges'], payload['max_piece_chars'])
        expected = [None if value is None else value.hex() for value in tokenizer.token_bytes]
        if payload.get('vocab_hex') != expected:
            raise ValueError('Vocabulary is inconsistent with the merge history')
        return tokenizer


In [7]:
class DecoderAttention(nn.Module):

    def __init__(self, width, heads, dropout):
        super().__init__()
        assert width % heads == 0
        self.heads, self.dim, self.dropout = (heads, width // heads, dropout)
        self.qkv = nn.Linear(width, 3 * width)
        self.proj = nn.Linear(width, width)

    def forward(self, x, past=None, cache=False):
        b, t, c = x.shape
        q, k, v = self.qkv(x).view(b, t, 3, self.heads, self.dim).unbind(2)
        q, k, v = (a.transpose(1, 2) for a in (q, k, v))
        if past is not None:
            if t != 1:
                raise ValueError('Cached decoding accepts one new token at a time')
            k, v = (torch.cat((past[0], k), 2), torch.cat((past[1], v), 2))
        y = F.scaled_dot_product_attention(q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=past is None)
        return (self.proj(y.transpose(1, 2).contiguous().view(b, t, c)), (k, v) if cache else None)


class Block(nn.Module):

    def __init__(self, width, heads, dropout):
        super().__init__()
        self.ln1, self.ln2 = (nn.LayerNorm(width), nn.LayerNorm(width))
        self.attn = DecoderAttention(width, heads, dropout)
        self.mlp = nn.Sequential(nn.Linear(width, 4 * width), nn.GELU(), nn.Linear(4 * width, width))
        self.drop = nn.Dropout(dropout)

    def forward(self, x, past=None, cache=False):
        a, kv = self.attn(self.ln1(x), past, cache)
        x = x + self.drop(a)
        return (x + self.drop(self.mlp(self.ln2(x))), kv)


class DecoderLM(nn.Module):

    def __init__(self, vocab_size=32768, width=704, layers=16, heads=11, context=2048, dropout=0.0):
        super().__init__()
        self.config = dict(vocab_size=vocab_size, width=width, layers=layers, heads=heads, context=context, dropout=dropout)
        self.token = nn.Embedding(vocab_size, width)
        self.position = nn.Embedding(context, width)
        self.blocks = nn.ModuleList([Block(width, heads, dropout) for _ in range(layers)])
        self.final_norm = nn.LayerNorm(width)
        self.drop = nn.Dropout(dropout)
        self.gradient_checkpointing = True
        self.apply(self._init)
        for block in self.blocks:
            nn.init.normal_(block.attn.proj.weight, std=0.02 / math.sqrt(2 * layers))
            nn.init.normal_(block.mlp[2].weight, std=0.02 / math.sqrt(2 * layers))

    @staticmethod
    def _init(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, ids, targets=None, past=None, cache=False):
        offset = 0 if past is None else past[0][0].shape[2]
        if offset + ids.shape[1] > self.config['context']:
            raise ValueError('Context window exceeded')
        x = self.drop(self.token(ids) + self.position(torch.arange(offset, offset + ids.shape[1], device=ids.device)))
        kvs = []
        for i, block in enumerate(self.blocks):
            if self.training and self.gradient_checkpointing:
                x = checkpoint(lambda z, b=block: b(z)[0], x, use_reentrant=False)
            else:
                x, kv = block(x, None if past is None else past[i], cache)
                if cache:
                    kvs.append(kv)
        x = self.final_norm(x)
        if targets is not None:
            flat, labels = (x.reshape(-1, x.shape[-1]), targets.reshape(-1))

            def chunk_loss(h, y):
                return F.cross_entropy(F.linear(h, self.token.weight).float(), y, ignore_index=-100, reduction='sum')
            loss = x.new_zeros((), dtype=torch.float32)
            for start in range(0, len(labels), 256):
                h, y = (flat[start:start + 256], labels[start:start + 256])
                loss = loss + (checkpoint(chunk_loss, h, y, use_reentrant=False) if self.training else chunk_loss(h, y))
            return loss
        return (F.linear(x[:, -1], self.token.weight), kvs)


In [8]:
class EncDecAttention(nn.Module):
    def __init__(self,width,heads,dropout=0.0):
        super().__init__(); assert width%heads==0
        self.heads=heads; self.dim=width//heads; self.dropout=dropout
        self.q=nn.Linear(width,width); self.k=nn.Linear(width,width)
        self.v=nn.Linear(width,width); self.out=nn.Linear(width,width)

    def forward(self,x,memory=None,valid=None,causal=False,past=None,cache=False):
        b,t,w=x.shape
        def split(z): return z.view(b,-1,self.heads,self.dim).transpose(1,2)
        q=split(self.q(x))
        if memory is not None and past is not None:
            k,v=past
        else:
            source=x if memory is None else memory
            k,v=split(self.k(source)),split(self.v(source))
            if past is not None:
                k=torch.cat((past[0],k),dim=2); v=torch.cat((past[1],v),dim=2)
        mask=None
        if valid is not None: mask=valid[:,None,None,:]
        if causal:
            offset=k.shape[2]-t
            allowed=torch.arange(k.shape[2],device=x.device)[None,:] <= (torch.arange(t,device=x.device)+offset)[:,None]
            mask=allowed[None,None,:,:] if mask is None else mask & allowed[None,None,:,:]
        h=F.scaled_dot_product_attention(q,k,v,attn_mask=mask,
            dropout_p=self.dropout if self.training else 0.0)
        h=self.out(h.transpose(1,2).contiguous().view(b,t,w))
        return h, (k,v) if cache else None

class EncoderBlock(nn.Module):
    def __init__(self,w,h,p):
        super().__init__(); self.n1=nn.LayerNorm(w); self.n2=nn.LayerNorm(w)
        self.attn=EncDecAttention(w,h,p); self.drop=nn.Dropout(p)
        self.ff=nn.Sequential(nn.Linear(w,4*w),nn.GELU(),nn.Linear(4*w,w))
    def forward(self,x,valid):
        x=x+self.drop(self.attn(self.n1(x),valid=valid)[0])
        return x+self.drop(self.ff(self.n2(x)))

class DecoderBlock(nn.Module):
    def __init__(self,w,h,p):
        super().__init__(); self.n1=nn.LayerNorm(w); self.n2=nn.LayerNorm(w); self.n3=nn.LayerNorm(w)
        self.self_attn=EncDecAttention(w,h,p); self.cross=EncDecAttention(w,h,p); self.drop=nn.Dropout(p)
        self.ff=nn.Sequential(nn.Linear(w,4*w),nn.GELU(),nn.Linear(4*w,w))
    def forward(self,x,memory,source_valid,target_valid=None,past=None,cache=False):
        self_past,cross_past=(None,None) if past is None else past
        a,skv=self.self_attn(self.n1(x),valid=target_valid,causal=True,past=self_past,cache=cache)
        x=x+self.drop(a)
        a,ckv=self.cross(self.n2(x),memory=memory,valid=source_valid,past=cross_past,cache=cache)
        x=x+self.drop(a); x=x+self.drop(self.ff(self.n3(x)))
        return x, (skv,ckv) if cache else None


class EncoderDecoder(nn.Module):
    def __init__(self,vocab_size=32768,width=704,encoder_layers=6,decoder_layers=8,heads=11,dropout=0.0):
        super().__init__()
        self.config=dict(vocab_size=vocab_size,width=width,encoder_layers=encoder_layers,
                         decoder_layers=decoder_layers,heads=heads,dropout=dropout)
        self.token=nn.Embedding(vocab_size,width)
        self.encoder=nn.ModuleList([EncoderBlock(width,heads,dropout) for _ in range(encoder_layers)])
        self.decoder=nn.ModuleList([DecoderBlock(width,heads,dropout) for _ in range(decoder_layers)])
        self.enc_norm=nn.LayerNorm(width); self.dec_norm=nn.LayerNorm(width)
        self.drop=nn.Dropout(dropout); self.gradient_checkpointing=True
        self.apply(self.initialize)

    @staticmethod
    def initialize(m):
        if isinstance(m,(nn.Linear,nn.Embedding)):
            nn.init.normal_(m.weight,std=0.02)
            if isinstance(m,nn.Linear) and m.bias is not None: nn.init.zeros_(m.bias)

    def embed(self,ids,offset=0):
        w=self.config['width']; positions=torch.arange(offset,offset+ids.shape[1],device=ids.device).float()
        frequency=torch.exp(torch.arange(0,w,2,device=ids.device).float()*(-math.log(10000)/w))
        angle=positions[:,None]*frequency[None,:]
        pe=torch.stack((angle.sin(),angle.cos()),dim=-1).flatten(1)[:,:w]
        return self.drop(self.token(ids)+pe.to(self.token.weight.dtype))

    def encode(self,src):
        valid=src.ne(PAD); x=self.embed(src)
        for block in self.encoder:
            if self.training and self.gradient_checkpointing:
                x=checkpoint(block,x,valid,use_reentrant=False)
            else: x=block(x,valid)
        return self.enc_norm(x),valid

    def decode(self,ids,memory,source_valid,past=None,cache=False):
        offset=0 if past is None else past[0][0][0].shape[2]
        x=self.embed(ids,offset)
        valid=ids.ne(PAD) if past is None else None
        caches=[]
        for i,block in enumerate(self.decoder):
            if self.training and self.gradient_checkpointing:
                x=checkpoint(lambda a,b,c,d,layer=block: layer(a,b,c,d)[0],
                    x,memory,source_valid,valid,use_reentrant=False)
            else:
                x,kv=block(x,memory,source_valid,valid,None if past is None else past[i],cache)
                if cache: caches.append(kv)
        return self.dec_norm(x),caches

    def forward(self,src,decoder_ids,labels=None):
        memory,valid=self.encode(src); h,_=self.decode(decoder_ids,memory,valid)
        if labels is None: return F.linear(h,self.token.weight)
        flat=h.reshape(-1,h.shape[-1]); labels=labels.reshape(-1)
        def loss_chunk(a,b):
            return F.cross_entropy(F.linear(a,self.token.weight).float(),b,ignore_index=-100,reduction='sum')
        loss=h.new_zeros((),dtype=torch.float32)
        for start in range(0,len(labels),128):
            a,b=flat[start:start+128],labels[start:start+128]
            loss=loss+(checkpoint(loss_chunk,a,b,use_reentrant=False) if self.training else loss_chunk(a,b))
        return loss


In [9]:
def sliding_attention(q,k,v,window,chunk,dropout=0.0):
    b,h,t,d=q.shape; blocks=math.ceil(t/chunk); padded=blocks*chunk
    qpad=F.pad(q,(0,0,0,padded-t))
    query=qpad.reshape(b,h,blocks,chunk,d).permute(0,2,1,3,4).reshape(b*blocks,h,chunk,d)
    positions=torch.arange(blocks,device=q.device)[:,None]*chunk-window+1+torch.arange(window+chunk-1,device=q.device)[None,:]
    indices=positions.clamp(0,t-1)
    keys=k[:,:,indices,:].permute(0,2,1,3,4).reshape(b*blocks,h,window+chunk-1,d)
    values=v[:,:,indices,:].permute(0,2,1,3,4).reshape(b*blocks,h,window+chunk-1,d)
    qi=torch.arange(chunk,device=q.device)[:,None]
    kj=torch.arange(window+chunk-1,device=q.device)[None,:]
    relative=(kj>=qi)&(kj<=qi+window-1)
    valid=(positions>=0)&(positions<t)
    mask=(relative[None,:,:]&valid[:,None,:])[None,:,:,:].expand(b,-1,-1,-1).reshape(b*blocks,1,chunk,window+chunk-1)
    output=F.scaled_dot_product_attention(query,keys,values,attn_mask=mask,dropout_p=dropout)
    return output.reshape(b,blocks,h,chunk,d).permute(0,2,1,3,4).reshape(b,h,padded,d)[:,:,:t]

class MixedAttention(nn.Module):
    def __init__(self,width,heads,window,query_chunk,global_layer,dropout):
        super().__init__(); assert width%heads==0
        self.heads=heads; self.dim=width//heads; self.window=window
        self.chunk=query_chunk; self.global_layer=global_layer; self.dropout=dropout
        self.qkv=nn.Linear(width,3*width); self.proj=nn.Linear(width,width)
    def forward(self,x,past=None,cache=False):
        b,t,w=x.shape
        q,k,v=self.qkv(x).view(b,t,3,self.heads,self.dim).unbind(2)
        q,k,v=(a.transpose(1,2) for a in (q,k,v))
        if past is not None:
            if t!=1: raise ValueError('Incremental decoding expects one token')
            k=torch.cat((past[0],k),dim=2); v=torch.cat((past[1],v),dim=2)
        p=self.dropout if self.training else 0.0
        if self.global_layer or past is not None:
            output=F.scaled_dot_product_attention(q,k,v,dropout_p=p,is_causal=past is None)
        else: output=sliding_attention(q,k,v,self.window,self.chunk,p)
        kv=None
        if cache:
            kv=(k,v) if self.global_layer else (k[:,:,-(self.window-1):].contiguous(),v[:,:,-(self.window-1):].contiguous())
        return self.proj(output.transpose(1,2).contiguous().view(b,t,w)),kv

class MixedBlock(nn.Module):
    def __init__(self,width,heads,window,query_chunk,global_layer,dropout):
        super().__init__(); self.n1=nn.LayerNorm(width); self.n2=nn.LayerNorm(width)
        self.attn=MixedAttention(width,heads,window,query_chunk,global_layer,dropout)
        self.ff=nn.Sequential(nn.Linear(width,4*width),nn.GELU(),nn.Linear(4*width,width))
        self.drop=nn.Dropout(dropout)
    def forward(self,x,past=None,cache=False):
        a,kv=self.attn(self.n1(x),past,cache); x=x+self.drop(a)
        return x+self.drop(self.ff(self.n2(x))),kv


class LocalGlobalLM(nn.Module):
    def __init__(self,vocab_size=32768,width=704,layers=16,heads=11,context=2048,window=256,
                 global_every=4,query_chunk=128,dropout=0.0,activation_checkpointing=True,loss_chunk=256):
        super().__init__()
        if min(context,query_chunk,global_every,loss_chunk)<1 or window<2: raise ValueError('Invalid architecture limits')
        self.config=dict(vocab_size=vocab_size,width=width,layers=layers,heads=heads,context=context,
            window=window,global_every=global_every,query_chunk=query_chunk,dropout=dropout,
            activation_checkpointing=activation_checkpointing,loss_chunk=loss_chunk)
        self.token=nn.Embedding(vocab_size,width)
        self.blocks=nn.ModuleList([MixedBlock(width,heads,window,query_chunk,(i+1)%global_every==0,dropout) for i in range(layers)])
        self.norm=nn.LayerNorm(width); self.drop=nn.Dropout(dropout)
        self.apply(self.initialize)
        for block in self.blocks:
            nn.init.normal_(block.attn.proj.weight,std=0.02/math.sqrt(2*layers))
            nn.init.normal_(block.ff[2].weight,std=0.02/math.sqrt(2*layers))
    @staticmethod
    def initialize(m):
        if isinstance(m,(nn.Linear,nn.Embedding)):
            nn.init.normal_(m.weight,std=0.02)
            if isinstance(m,nn.Linear) and m.bias is not None: nn.init.zeros_(m.bias)
    def hidden(self,ids,past=None,position_offset=0,cache=False):
        t=ids.shape[1]; w=self.config['width']
        if position_offset+t>self.config['context']: raise ValueError('Context exceeded')
        pos=torch.arange(position_offset,position_offset+t,device=ids.device).float()
        freq=torch.exp(torch.arange(0,w,2,device=ids.device).float()*(-math.log(10000)/w))
        phase=pos[:,None]*freq[None,:]
        pe=torch.stack((phase.sin(),phase.cos()),dim=-1).flatten(1)[:,:w]
        # Match sinusoidal amplitude with scaled learned token embeddings.
        x=self.drop(self.token(ids)*math.sqrt(w)+pe.to(self.token.weight.dtype))
        caches=[]
        for i,block in enumerate(self.blocks):
            if self.training and self.config['activation_checkpointing']:
                x=checkpoint(lambda z,layer=block: layer(z)[0],x,use_reentrant=False)
            else:
                x,kv=block(x,None if past is None else past[i],cache)
                if cache: caches.append(kv)
        return self.norm(x),caches
    def forward(self,ids,targets=None,past=None,position_offset=0,cache=False):
        h,caches=self.hidden(ids,past,position_offset,cache)
        if targets is None: return F.linear(h[:,-1],self.token.weight),caches
        flat=h.reshape(-1,h.shape[-1]); labels=targets.reshape(-1)
        def chunk_loss(a,b): return F.cross_entropy(F.linear(a,self.token.weight).float(),b,ignore_index=-100,reduction='sum')
        loss=h.new_zeros((),dtype=torch.float32); size=self.config['loss_chunk']
        for start in range(0,len(labels),size):
            a,b=flat[start:start+size],labels[start:start+size]
            loss=loss+(checkpoint(chunk_loss,a,b,use_reentrant=False) if self.training and self.config['activation_checkpointing'] else chunk_loss(a,b))
        return loss


In [10]:
class CausalGatedConv(nn.Module):
    def __init__(self,width,kernel_size,dilation):
        super().__init__()
        if kernel_size<2 or dilation<1: raise ValueError('Invalid causal convolution settings')
        self.history=(kernel_size-1)*dilation
        self.conv=nn.Conv1d(width,2*width,kernel_size,dilation=dilation,padding=0)
        self.proj=nn.Linear(width,width)

    def forward(self,x,past=None,cache=False):
        values=x.transpose(1,2)
        if past is None:
            padded=F.pad(values,(self.history,0))
        else:
            if past.shape!=(x.shape[0],x.shape[2],self.history):
                raise ValueError('Convolution cache shape mismatch')
            padded=torch.cat((past,values),dim=2)
        features=self.conv(padded)
        a,b=features.chunk(2,dim=1)
        output=self.proj((a*torch.sigmoid(b)).transpose(1,2))
        # clone prevents a small cache view from retaining a full prompt buffer.
        state=padded[:,:,-self.history:].clone() if cache else None
        return output,state

class ConvBlock(nn.Module):
    def __init__(self,width,kernel_size,dilation,dropout):
        super().__init__()
        self.n1=nn.LayerNorm(width); self.n2=nn.LayerNorm(width)
        self.causal=CausalGatedConv(width,kernel_size,dilation)
        self.ff=nn.Sequential(nn.Linear(width,4*width),nn.GELU(),nn.Linear(4*width,width))
        self.drop=nn.Dropout(dropout)

    def forward(self,x,past=None,cache=False):
        h,state=self.causal(self.n1(x),past,cache)
        x=x+self.drop(h)
        return x+self.drop(self.ff(self.n2(x))),state


class GatedConvLM(nn.Module):
    def __init__(self,vocab_size=32768,width=640,layers=16,kernel_size=3,dilation_cycle=10,
                 context=2048,dropout=0.0,activation_checkpointing=True,loss_chunk=256):
        super().__init__()
        if min(width,layers,dilation_cycle,context,loss_chunk)<1 or kernel_size<2:
            raise ValueError('Invalid model configuration')
        self.config=dict(vocab_size=vocab_size,width=width,layers=layers,kernel_size=kernel_size,
            dilation_cycle=dilation_cycle,context=context,dropout=dropout,
            activation_checkpointing=activation_checkpointing,loss_chunk=loss_chunk)
        self.dilations=[2**(i%dilation_cycle) for i in range(layers)]
        self.receptive_field=1+(kernel_size-1)*sum(self.dilations)
        self.token=nn.Embedding(vocab_size,width)
        self.blocks=nn.ModuleList([ConvBlock(width,kernel_size,d,dropout) for d in self.dilations])
        self.norm=nn.LayerNorm(width); self.drop=nn.Dropout(dropout)
        self.apply(self.initialize)
        for block in self.blocks:
            nn.init.normal_(block.causal.proj.weight,std=0.02/math.sqrt(2*layers))
            nn.init.normal_(block.ff[2].weight,std=0.02/math.sqrt(2*layers))

    @staticmethod
    def initialize(module):
        if isinstance(module,(nn.Linear,nn.Conv1d,nn.Embedding)):
            nn.init.normal_(module.weight,std=0.02)
            if getattr(module,'bias',None) is not None: nn.init.zeros_(module.bias)

    def hidden(self,ids,past=None,cache=False):
        if past is None and ids.shape[1]>self.config['context']:
            raise ValueError('Initial prompt/training block exceeds configured context')
        if past is not None and (len(past)!=len(self.blocks) or ids.shape[1]!=1):
            raise ValueError('Cached generation accepts one token and one state per layer')
        if self.training and (cache or past is not None):
            raise ValueError('Caches are for eval-mode generation only')
        x=self.drop(self.token(ids)); states=[]
        for i,block in enumerate(self.blocks):
            if self.training and self.config['activation_checkpointing']:
                x=checkpoint(lambda z,layer=block: layer(z)[0],x,use_reentrant=False)
            else:
                x,state=block(x,None if past is None else past[i],cache)
                if cache: states.append(state)
        return self.norm(x),states

    def forward(self,ids,targets=None,past=None,cache=False):
        h,states=self.hidden(ids,past,cache)
        if targets is None: return F.linear(h[:,-1],self.token.weight),states
        flat=h.reshape(-1,h.shape[-1]); labels=targets.reshape(-1)
        def chunk_loss(a,b):
            return F.cross_entropy(F.linear(a,self.token.weight).float(),b,ignore_index=-100,reduction='sum')
        loss=h.new_zeros((),dtype=torch.float32); size=self.config['loss_chunk']
        for start in range(0,len(labels),size):
            a,b=flat[start:start+size],labels[start:start+size]
            loss=loss+(checkpoint(chunk_loss,a,b,use_reentrant=False) if self.training and self.config['activation_checkpointing'] else chunk_loss(a,b))
        return loss


In [11]:
MODEL_CLASSES = {
    "model1": DecoderLM,
    "model2": EncoderDecoder,
    "model3": LocalGlobalLM,
    "model4": GatedConvLM,
}

BASES = {}
for name in MODEL_NAMES:
    folder = MODEL_DIRS[name]
    checkpoint_path = folder / "best.pt"
    tokenizer_path = folder / "tokenizer.json"
    state = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    config = copy.deepcopy(state["model_config"])
    detected = classify_model_config(config)
    if detected != name:
        raise ValueError(f"{folder} is {detected}, expected {name}")
    tokenizer = ScratchByteBPE.from_file(tokenizer_path)
    if config["vocab_size"] != tokenizer.get_vocab_size():
        raise ValueError(f"Vocabulary mismatch for {name}")
    expected_hash = state.get("tokenizer_sha256")
    if expected_hash and digest_file(tokenizer_path) != expected_hash:
        raise ValueError(f"Tokenizer hash mismatch for {name}")
    BASES[name] = {
        "folder": folder,
        "checkpoint": checkpoint_path,
        "tokenizer_path": tokenizer_path,
        "tokenizer": tokenizer,
        "config": config,
        "state": state["model"],
        "checkpoint_sha256": digest_file(checkpoint_path),
        "tokenizer_sha256": digest_file(tokenizer_path),
    }

base_report = {}
for name in MODEL_NAMES:
    model = MODEL_CLASSES[name](**BASES[name]["config"])
    result = model.load_state_dict(BASES[name]["state"], strict=True)
    assert not result.missing_keys and not result.unexpected_keys
    count = sum(parameter.numel() for parameter in model.parameters())
    base_report[name] = {"parameters": count, "config": BASES[name]["config"]}
    print(name, f"{count:,}", BASES[name]["config"])
    del model
    gc.collect()
write_json(ASSET_DIR / "phase1_models.json", base_report)


model1 119,816,576 {'vocab_size': 32768, 'width': 704, 'layers': 16, 'heads': 11, 'context': 2048, 'dropout': 0.0}
model2 122,356,608 {'vocab_size': 32768, 'width': 704, 'encoder_layers': 6, 'decoder_layers': 8, 'heads': 11, 'dropout': 0.0}
model3 118,374,784 {'vocab_size': 32768, 'width': 704, 'layers': 16, 'heads': 11, 'context': 2048, 'window': 256, 'global_every': 4, 'query_chunk': 128, 'dropout': 0.0, 'activation_checkpointing': False, 'loss_chunk': 256}
model4 119,399,680 {'vocab_size': 32768, 'width': 640, 'layers': 16, 'kernel_size': 3, 'dilation_cycle': 10, 'context': 2048, 'dropout': 0.0, 'activation_checkpointing': False, 'loss_chunk': 256}


In [12]:
for name in MODEL_NAMES:
    saved = torch.load(BASES[name]["checkpoint"], map_location="cpu", weights_only=False)
    print(name, {key: saved.get(key) for key in ("step", "tokens", "val_loss")})
    del saved
gc.collect()


model1 {'step': 6750, 'tokens': 442365850, 'val_loss': 2.0184897229003904}
model2 {'step': 19500, 'tokens': 499999202, 'val_loss': 3.4662523185995666}
model3 {'step': 4731, 'tokens': 310000000, 'val_loss': 2.0665987357497215}
model4 {'step': 7630, 'tokens': 500000000, 'val_loss': 2.4732237830758095}


30

## Data and sentence boundaries

The current sentence has its own mask. Pooling excludes BOS, EOS, padding and neighboring sentences. Context follows document order within each side, with a separator between the preceding and following groups. The original sentence and document order is retained for prediction files.


In [13]:
def load_label_vocab(path):
    payload = read_json(path)
    if isinstance(payload, list):
        labels = payload
    elif isinstance(payload, dict) and payload:
        values = list(payload.values())
        if all(isinstance(value, int) and not isinstance(value, bool) for value in values):
            labels = list(payload.keys())
        elif all(isinstance(value, str) for value in values):
            def key_order(item):
                try:
                    return int(item[0])
                except (TypeError, ValueError):
                    return str(item[0])
            labels = [value for _, value in sorted(payload.items(), key=key_order)]
        else:
            raise ValueError("Unsupported label_vocab.json format")
    else:
        raise ValueError("Empty label vocabulary")
    if len(labels) != 13 or len(set(labels)) != 13:
        raise ValueError(f"Expected 13 distinct labels, found {len(labels)}")
    return labels

def normalize_documents(path, domain, split_name, require_labels):
    payload = read_json(path)
    if isinstance(payload, dict) and "data" in payload and isinstance(payload["data"], list):
        payload = payload["data"]
    if isinstance(payload, dict):
        records = []
        for key, value in payload.items():
            if isinstance(value, dict):
                record = dict(value)
                record.setdefault("id", key)
            elif isinstance(value, list):
                record = {"id": key, "text": value}
            else:
                raise ValueError(f"Unsupported document entry in {path}: {key}")
            records.append(record)
    elif isinstance(payload, list):
        records = payload
    else:
        raise ValueError(f"Expected list or object in {path}")

    documents = []
    seen = set()
    for position, record in enumerate(records):
        if not isinstance(record, dict):
            raise ValueError(f"Document {position} in {path} is not an object")
        raw_id = record.get("id", record.get("document_id", record.get("doc_id")))
        if raw_id is None:
            raise ValueError(f"Document {position} in {path} has no ID")
        doc_id = normalize_id(raw_id)
        if doc_id in seen:
            raise ValueError(f"Duplicate document ID in {path}: {doc_id}")
        seen.add(doc_id)
        sentences = record.get("text", record.get("sentences"))
        if not isinstance(sentences, list) or any(not isinstance(text, str) for text in sentences):
            raise ValueError(f"{doc_id}: text must be a list of strings")
        labels = record.get("labels")
        if require_labels:
            if not isinstance(labels, list) or len(labels) != len(sentences):
                raise ValueError(f"{doc_id}: sentence/label mismatch")
        elif labels is not None:
            raise ValueError(f"{doc_id}: test labels must not be present")
        documents.append({
            "id": doc_id,
            "domain": domain,
            "split": split_name,
            "text": sentences,
            "labels": labels,
        })
    return documents

LABELS = load_label_vocab(DATA_ROOT / "Train/label_vocab.json")
LABEL_TO_ID = {label: index for index, label in enumerate(LABELS)}

TRAIN_DOCS = (
    normalize_documents(DATA_ROOT / "Train/CL_train.json", "CL", "train", True)
    + normalize_documents(DATA_ROOT / "Train/IT_train.json", "IT", "train", True)
)
DEV_DOCS = (
    normalize_documents(DATA_ROOT / "Train/CL_dev.json", "CL", "dev", True)
    + normalize_documents(DATA_ROOT / "Train/IT_dev.json", "IT", "dev", True)
)
TEST_DOCS = (
    normalize_documents(DATA_ROOT / "Test/CL_test.json", "CL", "test", False)
    + normalize_documents(DATA_ROOT / "Test/IT_test.json", "IT", "test", False)
)

for group_name, documents in (("train", TRAIN_DOCS), ("dev", DEV_DOCS), ("test", TEST_DOCS)):
    ids = [document["id"] for document in documents]
    if len(ids) != len(set(ids)):
        raise ValueError(f"Overlapping CL/IT IDs in {group_name}")
for document in TRAIN_DOCS + DEV_DOCS:
    unknown = set(document["labels"]) - set(LABELS)
    if unknown:
        raise ValueError(f"Unknown labels in {document['id']}: {unknown}")

DATA_FILES = {
    str(path.relative_to(DATA_ROOT)): digest_file(path)
    for path in sorted(DATA_ROOT.rglob("*.json"))
}
write_json(ASSET_DIR / "labels.json", LABELS)
write_json(ASSET_DIR / "dataset_hashes.json", DATA_FILES)

def corpus_report(documents):
    labels = Counter()
    domain_docs = Counter()
    domain_sentences = Counter()
    lengths = []
    for document in documents:
        domain_docs[document["domain"]] += 1
        domain_sentences[document["domain"]] += len(document["text"])
        lengths.append(len(document["text"]))
        if document["labels"] is not None:
            labels.update(document["labels"])
    return {
        "documents": len(documents),
        "sentences": sum(lengths),
        "documents_by_domain": dict(domain_docs),
        "sentences_by_domain": dict(domain_sentences),
        "sentence_count_percentiles": {
            str(p): float(np.percentile(lengths, p)) if lengths else 0.0
            for p in (50, 90, 95, 99, 100)
        },
        "labels": dict(labels),
    }

audit = {
    "train": corpus_report(TRAIN_DOCS),
    "dev": corpus_report(DEV_DOCS),
    "test": corpus_report(TEST_DOCS),
}
write_json(ASSET_DIR / "data_audit.json", audit)
print(json.dumps(audit, indent=2))


{
  "train": {
    "documents": 80,
    "sentences": 16845,
    "documents_by_domain": {
      "CL": 40,
      "IT": 40
    },
    "sentences_by_domain": {
      "CL": 10593,
      "IT": 6252
    },
    "sentence_count_percentiles": {
      "50": 123.5,
      "90": 413.0000000000001,
      "95": 760.05,
      "99": 1810.47,
      "100": 1816.0
    },
    "labels": {
      "Fact": 4098,
      "ArgumentPetitioner": 1613,
      "ArgumentRespondent": 1348,
      "RatioOfTheDecision": 4301,
      "RulingByPresentCourt": 307,
      "RulingByLowerCourt": 874,
      "None": 297,
      "PrecedentNotReliedUpon": 569,
      "PrecedentReliedUpon": 1945,
      "Statute": 1099,
      "Issue": 107,
      "Dissent": 284,
      "PrecedentOverruled": 3
    }
  },
  "dev": {
    "documents": 10,
    "sentences": 2142,
    "documents_by_domain": {
      "CL": 5,
      "IT": 5
    },
    "sentences_by_domain": {
      "CL": 1396,
      "IT": 746
    },
    "sentence_count_percentiles": {
      "50": 121.5,

In [14]:
def flatten_documents(documents):
    records = []
    for document in documents:
        for index in range(len(document["text"])):
            label = None if document["labels"] is None else LABEL_TO_ID[document["labels"][index]]
            records.append({"document": document, "sentence_index": index, "label": label})
    return records

TRAIN_RECORDS = flatten_documents(TRAIN_DOCS)
DEV_RECORDS = flatten_documents(DEV_DOCS)
TEST_RECORDS = flatten_documents(TEST_DOCS)

for left_name, left, right_name, right in (
    ("train", TRAIN_DOCS, "dev", DEV_DOCS),
    ("train", TRAIN_DOCS, "test", TEST_DOCS),
    ("dev", DEV_DOCS, "test", TEST_DOCS),
):
    overlap = {doc["id"] for doc in left} & {doc["id"] for doc in right}
    if overlap:
        raise ValueError(f"Document IDs overlap between {left_name} and {right_name}: {sorted(overlap)[:5]}")

def bounded_tokens(values, limit, keep_tail=False):
    if limit <= 0:
        return []
    return values[-limit:] if keep_tail else values[:limit]

def encode_record(record, tokenizer, config):
    document = record["document"]
    index = record["sentence_index"]
    maximum = config["max_length"]
    raw_current = tokenizer.encode(document["text"][index]).ids
    limit = min(maximum - 2, max(8, int(maximum * config["current_sentence_fraction"])))
    if len(raw_current) > limit and config["current_truncation"] == "head_tail":
        head_size = (limit + 1) // 2
        current = raw_current[:head_size] + raw_current[-(limit - head_size):]
    else:
        current = raw_current[:limit]

    previous, following = [], []
    for position in range(max(0, index - config["left_context_sentences"]), index):
        previous.extend(tokenizer.encode(document["text"][position]).ids + [EOS])
    stop = min(len(document["text"]), index + 1 + config["right_context_sentences"])
    for position in range(index + 1, stop):
        following.extend(tokenizer.encode(document["text"][position]).ids + [EOS])

    budget = maximum - len(current) - 2
    context = []
    if previous and following and budget >= 3:
        available = budget - 1
        previous_size = min(len(previous), (available + 1) // 2)
        following_size = min(len(following), available - previous_size)
        previous_size = min(len(previous), available - following_size)
        context = bounded_tokens(previous, previous_size, True) + [EOS] + bounded_tokens(following, following_size)
    elif previous:
        context = bounded_tokens(previous, budget, True)
    elif following:
        context = bounded_tokens(following, budget)

    ids = [BOS] + context + current + [EOS]
    current_mask = [False] * (1 + len(context)) + [True] * len(current) + [False]
    assert len(ids) == len(current_mask) <= maximum
    return dict(ids=ids, current_mask=current_mask, label=record["label"],
                document_id=document["id"], domain=document["domain"], sentence_index=index,
                raw_current_tokens=len(raw_current), kept_current_tokens=len(current))

def encoded_cache_signature(model_name):
    return dict(version="phase3-rr-boundaries-v2", tokenizer=BASES[model_name]["tokenizer_sha256"],
                dataset=DATA_FILES, settings={key: CONFIG[key] for key in
                ("max_length", "current_sentence_fraction", "left_context_sentences",
                 "right_context_sentences", "current_truncation")})

ENCODED = {}
for model_name in MODEL_NAMES:
    cache_path = ASSET_DIR / "encoded" / f"{model_name}.pt"
    signature = encoded_cache_signature(model_name)
    if cache_path.is_file():
        payload = torch.load(cache_path, map_location="cpu", weights_only=False)
        if payload["signature"] != signature:
            raise ValueError(f"Input settings changed for {model_name}; choose a new OUTPUT_ROOT")
    else:
        tokenizer = BASES[model_name]["tokenizer"]
        payload = dict(signature=signature)
        for split_name, records in (("train", TRAIN_RECORDS), ("dev", DEV_RECORDS), ("test", TEST_RECORDS)):
            payload[split_name] = [encode_record(record, tokenizer, CONFIG) for record in records]
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        torch.save(payload, cache_path)
    ENCODED[model_name] = payload
    audit_rows = {}
    for split_name in ("train", "dev", "test"):
        entries = payload[split_name]
        lengths = [len(entry["ids"]) for entry in entries]
        audit_rows[split_name] = dict(examples=len(entries), max_tokens=max(lengths, default=0),
            p95_tokens=float(np.percentile(lengths, 95)) if lengths else 0.0,
            truncated_sentences=sum(entry["raw_current_tokens"] > entry["kept_current_tokens"] for entry in entries),
            removed_current_tokens=sum(entry["raw_current_tokens"] - entry["kept_current_tokens"] for entry in entries))
    print(model_name, json.dumps(audit_rows))
    write_json(ASSET_DIR / "input_audits" / f"{model_name}.json", audit_rows)

class EncodedDataset(Dataset):
    def __init__(self, entries):
        self.entries = entries
    def __len__(self):
        return len(self.entries)
    def __getitem__(self, index):
        return index, self.entries[index]

def collate_batch(batch):
    indexes = torch.tensor([index for index, entry in batch], dtype=torch.long)
    entries = [entry for index, entry in batch]
    length = max(len(entry["ids"]) for entry in entries)
    ids = torch.full((len(entries), length), PAD, dtype=torch.long)
    mask = torch.zeros_like(ids, dtype=torch.bool)
    current_mask = torch.zeros_like(ids, dtype=torch.bool)
    for row, entry in enumerate(entries):
        size = len(entry["ids"])
        ids[row, :size] = torch.tensor(entry["ids"], dtype=torch.long)
        mask[row, :size] = True
        current_mask[row, :size] = torch.tensor(entry["current_mask"], dtype=torch.bool)
    labels = None if entries[0]["label"] is None else torch.tensor([entry["label"] for entry in entries])
    return indexes, ids, mask, current_mask, labels

counts = np.bincount([record["label"] for record in TRAIN_RECORDS], minlength=len(LABELS)).astype(np.float64)
weights = (counts.sum() / np.maximum(counts, 1.0)) ** CONFIG["class_weight_power"]
weights /= np.average(weights, weights=counts)
weights = np.minimum(weights, CONFIG["class_weight_cap"])
CLASS_WEIGHTS = torch.tensor(weights, dtype=torch.float32)
print({label: dict(count=int(counts[i]), weight=round(float(weights[i]), 4)) for i, label in enumerate(LABELS)})


model1 {"train": {"examples": 16845, "max_tokens": 512, "p95_tokens": 301.0, "truncated_sentences": 13, "removed_current_tokens": 3080}, "dev": {"examples": 2142, "max_tokens": 512, "p95_tokens": 293.0, "truncated_sentences": 2, "removed_current_tokens": 41}, "test": {"examples": 2197, "max_tokens": 512, "p95_tokens": 311.1999999999998, "truncated_sentences": 2, "removed_current_tokens": 291}}
model2 {"train": {"examples": 16845, "max_tokens": 512, "p95_tokens": 301.0, "truncated_sentences": 13, "removed_current_tokens": 3080}, "dev": {"examples": 2142, "max_tokens": 512, "p95_tokens": 293.0, "truncated_sentences": 2, "removed_current_tokens": 41}, "test": {"examples": 2197, "max_tokens": 512, "p95_tokens": 311.1999999999998, "truncated_sentences": 2, "removed_current_tokens": 291}}
model3 {"train": {"examples": 16845, "max_tokens": 512, "p95_tokens": 301.0, "truncated_sentences": 13, "removed_current_tokens": 3080}, "dev": {"examples": 2142, "max_tokens": 512, "p95_tokens": 293.0, "tr

## Adaptation and classifier

LoRA and LoRMA wrap the task-used linear layers. Model 2 adapts both encoder and decoder. LoRMA uses additive identity inflation on the output transformation; fused QKV projections have separate transformations for Q, K and V. IA³ scales key/value channels and feed-forward intermediate channels. In the convolution model it scales convolution gate channels and feed-forward intermediate channels. The classification head is trainable for every method.


In [15]:
class LoRALinear(nn.Module):
    def __init__(self, base, rank, alpha, rank_stabilized=False):
        super().__init__()
        self.base = base
        for parameter in self.base.parameters():
            parameter.requires_grad_(False)
        self.lora_A = nn.Parameter(torch.empty(rank, base.in_features))
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, rank))
        nn.init.normal_(self.lora_A, std=0.02)
        denominator = math.sqrt(rank) if rank_stabilized else rank
        self.scale = alpha / denominator
    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(x, self.lora_A), self.lora_B)

class LoRMALinear(nn.Module):
    def __init__(self, base, rank, alpha):
        super().__init__()
        self.base = base
        for parameter in self.base.parameters():
            parameter.requires_grad_(False)
        self.groups = 3 if base.out_features == 3 * base.in_features else 1
        self.group_width = base.out_features // self.groups
        self.lorma_A = nn.Parameter(torch.empty(self.groups, rank, self.group_width))
        self.lorma_B = nn.Parameter(torch.zeros(self.groups, self.group_width, rank))
        nn.init.normal_(self.lorma_A, std=0.02)
        self.scale = alpha / rank
    def forward(self, x):
        z = self.base(x)
        chunks = z.split(self.group_width, dim=-1)
        transformed = [
            chunk + self.scale * F.linear(F.linear(chunk, self.lorma_A[index]), self.lorma_B[index])
            for index, chunk in enumerate(chunks)
        ]
        return torch.cat(transformed, dim=-1)

class IA3Linear(nn.Module):
    def __init__(self, base, fused_qkv=False):
        super().__init__()
        self.base = base
        self.fused_qkv = fused_qkv
        for parameter in self.base.parameters():
            parameter.requires_grad_(False)
        size = 2 * base.in_features if fused_qkv else base.out_features
        self.ia3_scale = nn.Parameter(torch.ones(size))
    def forward(self, x):
        z = self.base(x)
        scale = torch.cat((torch.ones_like(self.ia3_scale[:self.base.in_features]), self.ia3_scale)) if self.fused_qkv else self.ia3_scale
        return z * scale

class IA3Conv(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base
        for parameter in self.base.parameters():
            parameter.requires_grad_(False)
        self.ia3_scale = nn.Parameter(torch.ones(base.out_channels))
    def forward(self, x):
        return self.base(x) * self.ia3_scale[None, :, None]

def replace_linears(module, method, rank, alpha, prefix=""):
    replaced = []
    for name, child in list(module.named_children()):
        path = f"{prefix}.{name}" if prefix else name
        if isinstance(child, nn.Linear):
            if method == "lora":
                wrapped = LoRALinear(child, rank, alpha)
            elif method == "lorma":
                wrapped = LoRMALinear(child, rank, alpha)
            elif method == "ia3":
                selected = name in ("qkv", "k", "v") or path.endswith(("ff.0", "mlp.0"))
                if not selected:
                    continue
                wrapped = IA3Linear(child, fused_qkv=name == "qkv")
            else:
                raise ValueError(method)
            setattr(module, name, wrapped)
            replaced.append(path)
        elif method == "ia3" and isinstance(child, nn.Conv1d):
            setattr(module, name, IA3Conv(child))
            replaced.append(path)
        else:
            replaced.extend(replace_linears(child, method, rank, alpha, path))
    return replaced

def set_activation_checkpointing(backbone, enabled):
    if hasattr(backbone, "gradient_checkpointing"):
        backbone.gradient_checkpointing = enabled
    if "activation_checkpointing" in backbone.config:
        backbone.config["activation_checkpointing"] = enabled

def decoder_hidden(backbone, ids):
    if ids.shape[1] > backbone.config["context"]:
        raise ValueError("Context window exceeded")
    positions = torch.arange(ids.shape[1], device=ids.device)
    x = backbone.drop(backbone.token(ids) + backbone.position(positions))
    for block in backbone.blocks:
        if backbone.training and backbone.gradient_checkpointing:
            x = checkpoint(lambda z, layer=block: layer(z)[0], x, use_reentrant=False)
        else:
            x = block(x)[0]
    return backbone.final_norm(x)

def masked_mean(hidden, selected, valid):
    selected = selected & valid
    selected = torch.where(selected.any(1, keepdim=True), selected, valid)
    weights = selected.unsqueeze(-1).to(torch.float32)
    return (hidden.float() * weights).sum(1) / weights.sum(1).clamp_min(1)

class RRClassifier(nn.Module):
    def __init__(self, model_name, method, config):
        super().__init__()
        if method not in DIRECT_METHODS:
            raise ValueError(method)
        base = BASES[model_name]
        self.model_name, self.method = model_name, method
        self.backbone = MODEL_CLASSES[model_name](**copy.deepcopy(base["config"]))
        self.backbone.load_state_dict(base["state"], strict=True)
        set_activation_checkpointing(self.backbone, config["activation_checkpointing"])
        self.replaced = []
        if method != "full":
            self.backbone.requires_grad_(False)
            targets = (self.backbone.encoder, self.backbone.decoder) if model_name == "model2" else (self.backbone.blocks,)
            for number, target in enumerate(targets):
                self.replaced.extend(replace_linears(target, method, config["rank"], config["alpha"], str(number)))
            if not self.replaced:
                raise ValueError(f"No adaptation modules for {model_name}/{method}")
        width = base["config"]["width"]
        with torch.random.fork_rng(devices=[]):
            torch.manual_seed(config["seed"] + 100)
            self.head_norm = nn.LayerNorm(2 * width)
            self.head_dropout = nn.Dropout(config["dropout"])
            self.classifier = nn.Linear(2 * width, len(LABELS))
            nn.init.normal_(self.classifier.weight, std=0.02)
            nn.init.zeros_(self.classifier.bias)

    def forward(self, ids, mask, current_mask):
        if self.model_name == "model2":
            hidden, source_valid = self.backbone.encode(ids)
            query = torch.full((len(ids), 1), BOS, dtype=torch.long, device=ids.device)
            decoded, _ = self.backbone.decode(query, hidden, source_valid)
            summary = decoded[:, -1].float()
        else:
            hidden = decoder_hidden(self.backbone, ids) if self.model_name == "model1" else self.backbone.hidden(ids)[0]
            final_positions = mask.long().sum(1) - 1
            summary = hidden[torch.arange(len(ids), device=ids.device), final_positions].float()
        current = masked_mean(hidden, current_mask, mask)
        pooled = torch.cat((current, summary), dim=-1)
        return self.classifier(self.head_dropout(self.head_norm(pooled)))

def make_model(model_name, method, config, device="cpu"):
    return RRClassifier(model_name, method, config).to(device)

def parameter_report(model):
    return dict(total=sum(p.numel() for p in model.parameters()),
                trainable=sum(p.numel() for p in model.parameters() if p.requires_grad),
                adapted_modules=len(model.replaced))


In [16]:
for model_name in MODEL_NAMES:
    for method in DIRECT_METHODS:
        seed_everything(CONFIG["seed"])
        probe = make_model(model_name, method, CONFIG)
        print(model_name, method, parameter_report(probe))
        del probe
    gc.collect()


model1 full {'total': 119837709, 'trainable': 119837709, 'adapted_modules': 0}
model1 lora {'total': 122721293, 'trainable': 2904717, 'adapted_modules': 64}
model1 lorma {'total': 123081741, 'trainable': 3265165, 'adapted_modules': 64}
model1 ia3 {'total': 119905293, 'trainable': 88717, 'adapted_modules': 32}
model2 full {'total': 122377741, 'trainable': 122377741, 'adapted_modules': 0}
model2 lora {'total': 125937165, 'trainable': 3580557, 'adapted_modules': 116}
model2 lorma {'total': 125937165, 'trainable': 3580557, 'adapted_modules': 116}
model2 ia3 {'total': 122448141, 'trainable': 91533, 'adapted_modules': 58}
model3 full {'total': 118395917, 'trainable': 118395917, 'adapted_modules': 0}
model3 lora {'total': 121279501, 'trainable': 2904717, 'adapted_modules': 64}
model3 lorma {'total': 121639949, 'trainable': 3265165, 'adapted_modules': 64}
model3 ia3 {'total': 118463501, 'trainable': 88717, 'adapted_modules': 32}
model4 full {'total': 119418893, 'trainable': 119418893, 'adapted

## Hardware check

Each training cell checks maximum-length forward, backward and optimizer updates. It compares batches of 1, 2, 4 and 8 examples per GPU, with checkpointing on and off, leaving at least 2 GiB of measured headroom. Timings include device synchronization. A CPU session stops before training.


In [17]:
def sync_cuda():
    if torch.cuda.is_available():
        for index in range(torch.cuda.device_count()):
            torch.cuda.synchronize(index)

def capacity_check(model_name, method, force=False):
    if not torch.cuda.is_available():
        raise RuntimeError("Enable the Kaggle GPU accelerator before training")
    result_path = ASSET_DIR / "hardware" / f"{model_name}__{method}.json"
    signature = dict(version="capacity-v2", checkpoint=BASES[model_name]["checkpoint_sha256"],
                     model=model_name, method=method, config=CONFIG,
                     gpus=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
    if result_path.is_file() and not force:
        saved = read_json(result_path)
        if saved["signature"] == signature:
            return saved["selected"]

    gpu_count = torch.cuda.device_count()
    results = []
    for checkpointing in (False, True):
        for micro in (1, 2, 4, 8):
            objects = {}
            try:
                gc.collect()
                torch.cuda.empty_cache()
                for index in range(gpu_count):
                    torch.cuda.reset_peak_memory_stats(index)
                trial = dict(CONFIG, activation_checkpointing=checkpointing)
                model = make_model(model_name, method, trial, "cuda:0")
                wrapped = nn.DataParallel(model) if gpu_count > 1 else model
                optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4)
                objects.update(model=model, wrapped=wrapped, optimizer=optimizer)
                count, length = micro * gpu_count, CONFIG["max_length"]
                ids = torch.randint(4, BASES[model_name]["config"]["vocab_size"], (count, length), device="cuda:0")
                mask = torch.ones_like(ids, dtype=torch.bool)
                current = torch.zeros_like(mask)
                current[:, length // 2:-1] = True
                labels = torch.randint(0, len(LABELS), (count,), device="cuda:0")
                # Use a modest initial scale for the capacity probe.
                scaler = torch.amp.GradScaler("cuda", init_scale=128)
                timings = []
                successful = 0
                for repetition in range(3):
                    optimizer.zero_grad(set_to_none=True)
                    sync_cuda()
                    start = time.perf_counter()
                    with torch.autocast("cuda", dtype=torch.float16):
                        logits = wrapped(ids, mask, current)
                        loss = F.cross_entropy(logits.float(), labels)
                    if not torch.isfinite(loss):
                        raise FloatingPointError("Nonfinite loss in the capacity check")
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    grad_norm = torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], CONFIG["clip"])
                    if not torch.isfinite(grad_norm):
                        raise FloatingPointError("Nonfinite gradient in the capacity check")
                    scaler.step(optimizer)
                    scaler.update()
                    sync_cuda()
                    successful += 1
                    if repetition:
                        timings.append(time.perf_counter() - start)
                memory = []
                for index in range(gpu_count):
                    free, total = torch.cuda.mem_get_info(index)
                    reserved = torch.cuda.max_memory_reserved(index)
                    free_at_peak = free + torch.cuda.memory_reserved(index) - reserved
                    memory.append(dict(gpu=index, peak_reserved_gib=reserved / 2**30,
                                       free_gib=max(0, free_at_peak) / 2**30, total_gib=total / 2**30))
                seconds = float(np.median(timings))
                row = dict(micro_batch_per_gpu=micro, activation_checkpointing=checkpointing,
                           seconds=seconds, examples_per_second=count / seconds,
                           projected_update_seconds=seconds * math.ceil(CONFIG["effective_batch"] / count),
                           memory=memory, successful_updates=successful, passed=True)
                results.append(row)
                print(model_name, method, dict(batch_per_gpu=micro, checkpointing=checkpointing,
                      seconds=round(seconds, 3), min_free_gib=round(min(m["free_gib"] for m in memory), 2)), flush=True)
            except RuntimeError as error:
                if "out of memory" not in str(error).lower():
                    raise
                results.append(dict(micro_batch_per_gpu=micro, activation_checkpointing=checkpointing,
                                    passed=False, error="out of memory"))
                print(model_name, method, micro, checkpointing, "out of memory", flush=True)
                break
            finally:
                objects.clear()
                model = wrapped = optimizer = ids = mask = current = labels = logits = loss = grad_norm = scaler = None
                gc.collect()
                torch.cuda.empty_cache()
    safe = [row for row in results if row.get("passed") and
            min(item["free_gib"] for item in row["memory"]) >= CONFIG["capacity_headroom_gib"]]
    if not safe:
        write_json(result_path, dict(signature=signature, results=results, selected=None))
        raise RuntimeError("No batch passed with the requested memory headroom; inspect the hardware report")
    selected = min(safe, key=lambda row: row["projected_update_seconds"])
    write_json(result_path, dict(signature=signature, results=results, selected=selected))
    print("Selected:", model_name, method, selected["micro_batch_per_gpu"], "examples/GPU", flush=True)
    return selected


## Training and evaluation helpers


In [18]:
def experiment_folder(model_name, method):
    return OUTPUT_ROOT / "direct" / model_name / method

def experiment_signature(model_name, method, config):
    return {
        "version": "phase3-rr-boundaries-pooling-v2",
        "model_name": model_name,
        "method": method,
        "phase1_checkpoint": BASES[model_name]["checkpoint_sha256"],
        "tokenizer": BASES[model_name]["tokenizer_sha256"],
        "dataset": DATA_FILES,
        "labels": LABELS,
        "config": config,
        "evaluator": digest_file(EVALUATOR_PATH),
        "input_signature": encoded_cache_signature(model_name),
    }

def split_metrics(y_true, y_pred):
    return {
        "macro_f1": float(f1_score(y_true, y_pred, labels=list(range(len(LABELS))), average="macro", zero_division=0)),
        "macro_precision": float(precision_score(y_true, y_pred, labels=list(range(len(LABELS))), average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, labels=list(range(len(LABELS))), average="macro", zero_division=0)),
    }

def predictions_from_ids(records, predicted_ids):
    output = {}
    for record, prediction in zip(records, predicted_ids):
        document = record["document"]
        if document["id"] not in output:
            output[document["id"]] = [None] * len(document["text"])
        slot = record["sentence_index"]
        if output[document["id"]][slot] is not None:
            raise ValueError("Duplicate sentence prediction")
        output[document["id"]][slot] = LABELS[int(prediction)]
    if any(value is None for labels in output.values() for value in labels):
        raise ValueError("Missing sentence prediction")
    return output

@torch.inference_mode()
def infer_logits(model, entries, batch_size, device):
    model.eval()
    dataset = EncodedDataset(entries)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_batch, num_workers=CONFIG["num_workers"], pin_memory=device.type == "cuda")
    outputs = []
    for _, ids, mask, current_mask, _ in loader:
        ids = ids.to(device, non_blocking=True)
        mask = mask.to(device, non_blocking=True)
        with torch.autocast(device.type, dtype=torch.float16, enabled=device.type == "cuda"):
            logits = model(ids, mask, current_mask.to(device, non_blocking=True))
        if not torch.isfinite(logits).all():
            raise FloatingPointError("Nonfinite inference logits")
        outputs.append(logits.float().cpu())
    return torch.cat(outputs)

def evaluate_model(model, model_name, batch_size, device, output_dir=None):
    logits = infer_logits(model, ENCODED[model_name]["dev"], batch_size, device)
    predicted = logits.argmax(-1).tolist()
    gold = [entry["label"] for entry in ENCODED[model_name]["dev"]]
    overall = split_metrics(gold, predicted)
    domains = {}
    for domain in ("CL", "IT"):
        indexes = [i for i, record in enumerate(DEV_RECORDS) if record["document"]["domain"] == domain]
        domains[domain] = split_metrics([gold[i] for i in indexes], [predicted[i] for i in indexes])
    per_label = {}
    for label_id, label in enumerate(LABELS):
        per_label[label] = {
            "f1": float(f1_score(gold, predicted, labels=[label_id], average="macro", zero_division=0)),
            "precision": float(precision_score(gold, predicted, labels=[label_id], average="macro", zero_division=0)),
            "recall": float(recall_score(gold, predicted, labels=[label_id], average="macro", zero_division=0)),
        }
    report = {"overall": overall, "domains": domains, "per_label": per_label}
    if output_dir is not None:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        predictions = predictions_from_ids(DEV_RECORDS, predicted)
        write_json(output_dir / "prediction.json", predictions)
        write_json(output_dir / "metrics.json", report)
    return report

def trainable_state(model):
    names = {name for name, parameter in model.named_parameters() if parameter.requires_grad}
    return {name: value.detach().cpu() for name, value in model.state_dict().items() if name in names}

def load_task_state(model, payload):
    state = payload["model"]
    if payload["storage"] == "full":
        result = model.load_state_dict(state, strict=True)
        if result.missing_keys or result.unexpected_keys:
            raise ValueError("Full checkpoint mismatch")
    else:
        result = model.load_state_dict(state, strict=False)
        expected = {name for name, parameter in model.named_parameters() if not parameter.requires_grad}
        if set(result.missing_keys) != expected or result.unexpected_keys:
            raise ValueError("Delta checkpoint mismatch")

def save_checkpoint(path, model, optimizer, scheduler, scaler, progress, signature, report, include_optimizer):
    plain = model.module if isinstance(model, nn.DataParallel) else model
    storage = "full" if signature["method"] in ("full", "kd") else "delta"
    payload = {
        "signature": signature,
        "storage": storage,
        "model": plain.state_dict() if storage == "full" else trainable_state(plain),
        "progress": progress,
        "report": report,
    }
    if include_optimizer:
        payload.update({
            "rng_state": capture_rng_state(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(),
        })
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, temporary)
    temporary.replace(path)

def validate_with_professor(run_dir, predictions_path):
    truth_cl = {doc["id"]: doc["labels"] for doc in DEV_DOCS if doc["domain"] == "CL"}
    truth_it = {doc["id"]: doc["labels"] for doc in DEV_DOCS if doc["domain"] == "IT"}
    write_json(run_dir / "dev_truth_cl.json", truth_cl)
    write_json(run_dir / "dev_truth_it.json", truth_it)
    labels_path = run_dir / "labels.json"
    write_json(labels_path, LABELS)
    command = [
        sys.executable, str(EVALUATOR_PATH),
        "--truth-cl", str(run_dir / "dev_truth_cl.json"),
        "--truth-it", str(run_dir / "dev_truth_it.json"),
        "--prediction", str(predictions_path),
        "--labels", str(labels_path),
    ]
    result = subprocess.run(command, capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    (predictions_path.parent / "prof_evaluation.txt").write_text(result.stdout + "\n" + result.stderr, encoding="utf-8")
    result.check_returncode()

def learning_rate_lambda(step, total_steps, warmup_fraction, min_ratio):
    warmup = max(1, int(total_steps * warmup_fraction))
    if step < warmup:
        return (step + 1) / warmup
    progress = min(1.0, (step - warmup) / max(1, total_steps - warmup))
    return min_ratio + (1 - min_ratio) * 0.5 * (1 + math.cos(math.pi * progress))

def capture_rng_state():
    return dict(python=random.getstate(), numpy=np.random.get_state(), torch=torch.get_rng_state(),
                cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None)

def restore_rng_state(state):
    if state is None:
        return
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"].cpu())
    if state["cuda"] is not None and torch.cuda.is_available():
        if len(state["cuda"]) == torch.cuda.device_count():
            torch.cuda.set_rng_state_all([value.cpu() for value in state["cuda"]])

def append_metric(path, value):
    with open(path, "a", encoding="utf-8") as handle:
        handle.write(json.dumps(value) + "\n")


In [19]:
def train_experiment(model_name, method, teacher_logits=None, teacher_reference=None):
    if method not in DIRECT_METHODS and method != "kd":
        raise ValueError(method)
    if not CONFIG["save_optimizer"]:
        raise ValueError("Keep save_optimizer=True for resumable training")
    run_dir = experiment_folder(model_name, method) if method != "kd" else OUTPUT_ROOT / "distillation" / model_name
    run_dir.mkdir(parents=True, exist_ok=True)
    base_method = "full" if method == "kd" else method
    hardware = capacity_check(model_name, base_method)
    run_config = dict(CONFIG, micro_batch_per_gpu=hardware["micro_batch_per_gpu"],
                      activation_checkpointing=hardware["activation_checkpointing"], teacher_reference=teacher_reference)
    run_config["inference_batch_size"] = min(CONFIG["inference_batch_size"], hardware["micro_batch_per_gpu"])
    signature = experiment_signature(model_name, method, run_config)
    signature_path = run_dir / "config.json"
    if signature_path.is_file() and read_json(signature_path) != signature:
        raise ValueError(f"Settings changed for {run_dir}; choose a new OUTPUT_ROOT")
    write_json(signature_path, signature)
    if (run_dir / "summary.json").is_file():
        completed = read_json(run_dir / "summary.json")
        if completed.get("complete"):
            print("Already complete:", run_dir, flush=True)
            return completed

    seed_everything(run_config["seed"])
    device = torch.device("cuda:0")
    gpu_count = torch.cuda.device_count()
    model = make_model(model_name, base_method, run_config, device)
    trainable = [parameter for parameter in model.parameters() if parameter.requires_grad]
    backbone = [parameter for name, parameter in model.named_parameters()
                if parameter.requires_grad and not name.startswith(("head_norm.", "classifier."))]
    head = [parameter for name, parameter in model.named_parameters()
            if parameter.requires_grad and name.startswith(("head_norm.", "classifier."))]
    backbone_lr = run_config["full_lr"] if base_method == "full" else run_config["peft_lr"]
    groups = [{"params": backbone, "lr": backbone_lr}, {"params": head, "lr": run_config["head_lr"]}]
    optimizer = torch.optim.AdamW([group for group in groups if group["params"]], weight_decay=run_config["weight_decay"])
    per_batch = run_config["micro_batch_per_gpu"] * gpu_count
    accumulation = max(1, math.ceil(run_config["effective_batch"] / per_batch))
    updates_per_epoch = math.ceil(math.ceil(len(ENCODED[model_name]["train"]) / per_batch) / accumulation)
    total_steps = updates_per_epoch * run_config["epochs"]
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer,
        lambda step: learning_rate_lambda(step, total_steps, run_config["warmup_fraction"], run_config["min_lr_ratio"]))
    scaler = torch.amp.GradScaler("cuda", init_scale=128)
    progress = dict(epoch=0, batch=0, step=0, best_macro_f1=-1.0, bad_epochs=0,
                    skipped_updates=0, complete=False)
    latest_path = run_dir / "latest.pt"
    pending_rng = None
    if latest_path.is_file():
        saved = torch.load(latest_path, map_location="cpu", weights_only=False)
        if saved["signature"] != signature:
            raise ValueError("Resume checkpoint settings differ")
        load_task_state(model, saved)
        optimizer.load_state_dict(saved["optimizer"])
        scheduler.load_state_dict(saved["scheduler"])
        scaler.load_state_dict(saved["scaler"])
        progress = copy.deepcopy(saved["progress"])
        pending_rng = saved.get("rng_state")
        del saved
        print("Resuming:", progress, flush=True)

    wrapped = nn.DataParallel(model) if gpu_count > 1 else model
    class_weights = CLASS_WEIGHTS.to(device)
    metrics_path = run_dir / "metrics.jsonl"
    started = time.perf_counter()
    loss_sum, weight_sum = 0.0, 0.0
    last_report = dict(parameters=parameter_report(model))
    checkpoint_epoch, checkpoint_batch = progress["epoch"], progress["batch"]
    print(model_name, method, parameter_report(model), "updates/epoch:", updates_per_epoch,
          "effective batch:", per_batch * accumulation, flush=True)

    def save_latest():
        save_checkpoint(latest_path, wrapped, optimizer, scheduler, scaler, copy.deepcopy(progress),
                        signature, last_report, True)

    if not latest_path.is_file():
        save_latest()
    try:
        for epoch in range(progress["epoch"], run_config["epochs"]):
            generator = torch.Generator().manual_seed(run_config["seed"] + epoch)
            loader = DataLoader(EncodedDataset(ENCODED[model_name]["train"]), batch_size=per_batch,
                shuffle=True, generator=generator, collate_fn=collate_batch,
                num_workers=run_config["num_workers"], pin_memory=True)
            model.train()
            optimizer.zero_grad(set_to_none=True)
            pending, normalizer = 0, 0.0
            for batch_index, (indexes, ids, mask, current_mask, labels) in enumerate(loader):
                if epoch == progress["epoch"] and batch_index < progress["batch"]:
                    continue
                if pending_rng is not None:
                    restore_rng_state(pending_rng)
                    pending_rng = None
                ids, mask, current_mask, labels = [value.to(device, non_blocking=True)
                                                  for value in (ids, mask, current_mask, labels)]
                with torch.autocast("cuda", dtype=torch.float16):
                    logits = wrapped(ids, mask, current_mask)
                    hard = F.cross_entropy(logits.float(), labels, reduction="none",
                                           label_smoothing=run_config["label_smoothing"])
                    if method == "kd":
                        if teacher_logits is None or len(teacher_logits) != len(TRAIN_RECORDS):
                            raise ValueError("Missing or misaligned teacher logits")
                        temperature = run_config["kd_temperature"]
                        soft = teacher_logits[indexes].to(device).float()
                        soft_loss = F.kl_div(F.log_softmax(logits.float() / temperature, dim=-1),
                            F.softmax(soft / temperature, dim=-1), reduction="none").sum(-1) * temperature**2
                        per_example = (1 - run_config["kd_weight"]) * hard + run_config["kd_weight"] * soft_loss
                    else:
                        per_example = hard
                    sample_weights = class_weights[labels]
                    objective_sum = (per_example * sample_weights).sum()
                if not torch.isfinite(objective_sum):
                    raise FloatingPointError("Nonfinite training loss; resume from latest.pt after inspection")
                scaler.scale(objective_sum).backward()
                batch_weight = float(sample_weights.sum())
                normalizer += batch_weight
                loss_sum += float(objective_sum.detach())
                weight_sum += batch_weight
                pending += 1
                if pending != accumulation and batch_index + 1 != len(loader):
                    continue

                scaler.unscale_(optimizer)
                for parameter in trainable:
                    if parameter.grad is not None:
                        parameter.grad.div_(max(normalizer, 1e-12))
                gradient_norm = torch.nn.utils.clip_grad_norm_(trainable, run_config["clip"])
                previous_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                skipped = scaler.get_scale() < previous_scale
                optimizer.zero_grad(set_to_none=True)
                pending, normalizer = 0, 0.0
                progress["batch"] = batch_index + 1
                if skipped:
                    progress["skipped_updates"] += 1
                    print("Skipped AMP update:", progress["skipped_updates"], "scale:", scaler.get_scale(), flush=True)
                    if progress["skipped_updates"] > run_config["max_skipped_updates"]:
                        raise FloatingPointError("Too many skipped optimizer updates; inspect the saved run")
                else:
                    scheduler.step()
                    progress["step"] += 1
                    if progress["step"] == 1 or progress["step"] % run_config["log_every"] == 0:
                        row = dict(step=progress["step"], epoch=epoch, loss=loss_sum / max(weight_sum, 1e-12),
                            lr=optimizer.param_groups[0]["lr"], gradient_norm=float(gradient_norm),
                            skipped_updates=progress["skipped_updates"], elapsed_minutes=(time.perf_counter() - started) / 60)
                        append_metric(metrics_path, row)
                        print(model_name, method, row, flush=True)
                        loss_sum, weight_sum = 0.0, 0.0
                if progress["step"] % run_config["save_every"] == 0 or skipped:
                    save_latest()
                    checkpoint_epoch, checkpoint_batch = progress["epoch"], progress["batch"]
                if (time.perf_counter() - started) / 3600 >= run_config["max_hours_per_call"]:
                    save_latest()
                    print("Session limit reached; rerun this training cell to resume", flush=True)
                    return dict(run_dir=str(run_dir), complete=False, progress=copy.deepcopy(progress))

            if pending_rng is not None:
                restore_rng_state(pending_rng)
                pending_rng = None
            report = evaluate_model(model, model_name, run_config["inference_batch_size"], device,
                                    run_dir / "validation" / f"epoch_{epoch + 1:02d}")
            last_report = report
            score = report["overall"]["macro_f1"]
            progress["epoch"], progress["batch"] = epoch + 1, 0
            append_metric(metrics_path, dict(step=progress["step"], epoch=epoch + 1, validation=report))
            print(model_name, method, "epoch", epoch + 1, "Macro-F1", round(score, 6),
                  "CL", round(report["domains"]["CL"]["macro_f1"], 6),
                  "IT", round(report["domains"]["IT"]["macro_f1"], 6), flush=True)
            if score > progress["best_macro_f1"]:
                progress["best_macro_f1"], progress["bad_epochs"] = score, 0
                save_checkpoint(run_dir / "best.pt", wrapped, optimizer, scheduler, scaler,
                                copy.deepcopy(progress), signature, report, False)
            else:
                progress["bad_epochs"] += 1
            save_latest()
            checkpoint_epoch, checkpoint_batch = progress["epoch"], progress["batch"]
            if progress["bad_epochs"] >= run_config["early_stopping_patience"]:
                print("Stopped after", progress["bad_epochs"], "epochs without improvement", flush=True)
                break

        progress["complete"] = True
        save_latest()
        best = torch.load(run_dir / "best.pt", map_location="cpu", weights_only=False)
        summary = dict(run_dir=str(run_dir), model_name=model_name, method=method,
            best_macro_f1=best["report"]["overall"]["macro_f1"], domains=best["report"]["domains"],
            parameters=parameter_report(model), best_checkpoint=str(run_dir / "best.pt"), complete=True)
        write_json(run_dir / "summary.json", summary)
        return summary
    except KeyboardInterrupt:
        print("Interrupted. Resume checkpoint:", latest_path, flush=True)
        print("Last saved epoch/batch:", checkpoint_epoch, checkpoint_batch, flush=True)
        raise
    finally:
        optimizer.zero_grad(set_to_none=True)
        del wrapped, model, trainable, backbone, head, groups, optimizer, scheduler, scaler
        gc.collect()
        torch.cuda.empty_cache()


In [20]:
def load_best_model(model_name, method, run_dir, device):
    run_dir = Path(run_dir)
    config = read_json(run_dir / "config.json")["config"]
    if encoded_cache_signature(model_name) != read_json(run_dir / "config.json")["input_signature"]:
        raise ValueError("Checkpoint inputs differ from the current encoded data")
    model = make_model(model_name, "full" if method == "kd" else method, config, device)
    payload = torch.load(run_dir / "best.pt", map_location="cpu", weights_only=False)
    load_task_state(model, payload)
    model.eval()
    return model, config

def run_folder(model_name, method):
    return OUTPUT_ROOT / "distillation" / model_name if method == "kd" else experiment_folder(model_name, method)

def validate_experiment(model_name, method):
    folder = run_folder(model_name, method)
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model, config = load_best_model(model_name, method, folder, device)
    try:
        output = folder / "best_validation"
        report = evaluate_model(model, model_name, config["inference_batch_size"], device, output)
        saved = torch.load(folder / "best.pt", map_location="cpu", weights_only=False)
        saved_score = saved["report"]["overall"]["macro_f1"]
        difference = abs(report["overall"]["macro_f1"] - saved_score)
        print("Saved F1:", saved_score, "Reloaded F1:", report["overall"]["macro_f1"], flush=True)
        if difference > 0.002:
            raise ValueError("Reloaded score changed materially; inspect before exporting predictions")
        write_json(output / "reproduction.json", dict(saved_f1=saved_score,
                   reloaded_f1=report["overall"]["macro_f1"], absolute_difference=difference,
                   checkpoint_sha256=digest_file(folder / "best.pt")))
        validate_with_professor(output, output / "prediction.json")
        for label in LABELS:
            print(label, report["per_label"][label])
        return report
    finally:
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

def validate_test_predictions(predictions):
    expected = {document["id"]: len(document["text"]) for document in TEST_DOCS}
    if set(predictions) != set(expected):
        raise ValueError("Official test document IDs do not match")
    for doc_id, values in predictions.items():
        if not isinstance(values, list) or len(values) != expected[doc_id] or not set(values) <= set(LABELS):
            raise ValueError(f"Invalid sentence predictions for {doc_id}")

def download_submission(archive):
    archive = Path(archive)
    payload = base64.b64encode(archive.read_bytes()).decode("ascii")
    name = html.escape(archive.name, quote=True)
    display(HTML(f'<a download="{name}" href="data:application/zip;base64,{payload}">Download {name}</a>'))

def export_experiment(model_name, method):
    folder = run_folder(model_name, method)
    if not (folder / "best_validation" / "prof_evaluation.txt").is_file():
        raise FileNotFoundError("Run this experiment's validation cell before exporting")
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model, config = load_best_model(model_name, method, folder, device)
    try:
        logits = infer_logits(model, ENCODED[model_name]["test"], config["inference_batch_size"], device)
        predictions = predictions_from_ids(TEST_RECORDS, logits.argmax(-1).tolist())
        validate_test_predictions(predictions)
        output = folder / "test_predictions"
        output.mkdir(parents=True, exist_ok=True)
        write_json(output / "prediction.json", predictions)
        write_json(output / "provenance.json", dict(checkpoint_sha256=digest_file(folder / "best.pt"),
                   tokenizer_sha256=BASES[model_name]["tokenizer_sha256"], input_signature=encoded_cache_signature(model_name),
                   documents=len(predictions), sentences=sum(map(len, predictions.values()))))
        torch.save(logits, output / "logits.pt")
        archive = output / f"submission_{model_name}_{method}_v2.zip"
        with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as handle:
            handle.write(output / "prediction.json", "prediction.json")
        with zipfile.ZipFile(archive) as handle:
            if handle.namelist() != ["prediction.json"] or handle.testzip() is not None:
                raise ValueError("Submission ZIP verification failed")
        print(archive, flush=True)
        download_submission(archive)
        return archive
    finally:
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

def require_complete(result):
    if not result.get("complete"):
        raise RuntimeError("Training paused. Rerun its training cell to finish before validation or export.")


## Experiments

For each selected experiment, run training, validation, and test export in order. Training resumes from that experiment's latest checkpoint. Validation and test export use its best development checkpoint.

## Model 1: standard decoder


### LoRA

Artifacts: `phase3_rr_v2/direct/model1/lora/`


In [21]:
model1_lora_result = train_experiment("model1", "lora")
print(model1_lora_result)


model1 lora {'batch_per_gpu': 1, 'checkpointing': False, 'seconds': 0.166, 'min_free_gib': 12.68}
model1 lora {'batch_per_gpu': 2, 'checkpointing': False, 'seconds': 0.17, 'min_free_gib': 12.72}
model1 lora {'batch_per_gpu': 4, 'checkpointing': False, 'seconds': 0.209, 'min_free_gib': 12.57}
model1 lora {'batch_per_gpu': 8, 'checkpointing': False, 'seconds': 0.286, 'min_free_gib': 11.79}
model1 lora {'batch_per_gpu': 1, 'checkpointing': True, 'seconds': 0.259, 'min_free_gib': 12.72}
model1 lora {'batch_per_gpu': 2, 'checkpointing': True, 'seconds': 0.24, 'min_free_gib': 12.73}
model1 lora {'batch_per_gpu': 4, 'checkpointing': True, 'seconds': 0.246, 'min_free_gib': 12.72}
model1 lora {'batch_per_gpu': 8, 'checkpointing': True, 'seconds': 0.362, 'min_free_gib': 12.71}
Selected: model1 lora 8 examples/GPU
model1 lora {'total': 122721293, 'trainable': 2904717, 'adapted_modules': 64} updates/epoch: 527 effective batch: 32
model1 lora {'step': 1, 'epoch': 0, 'loss': 2.7738072100802666, 'lr'

In [22]:
require_complete(model1_lora_result)
model1_lora_validation = validate_experiment("model1", "lora")


Saved F1: 0.3776417346669092 Reloaded F1: 0.3776417346669092
CL documents: 5
IT documents: 5
Total documents: 10
Total sentences: 2142
Macro-F1:        0.377642
Macro-Precision: 0.397954
Macro-Recall:    0.381044

Fact {'f1': 0.6446469248291572, 'precision': 0.652073732718894, 'recall': 0.6373873873873874}
Issue {'f1': 0.5517241379310345, 'precision': 0.5714285714285714, 'recall': 0.5333333333333333}
ArgumentPetitioner {'f1': 0.4294294294294294, 'precision': 0.5375939849624061, 'recall': 0.3575}
ArgumentRespondent {'f1': 0.26339285714285715, 'precision': 0.3352272727272727, 'recall': 0.21691176470588236}
Statute {'f1': 0.7441860465116279, 'precision': 0.6956521739130435, 'recall': 0.8}
Dissent {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
PrecedentReliedUpon {'f1': 0.4550898203592814, 'precision': 0.46153846153846156, 'recall': 0.44881889763779526}
PrecedentNotReliedUpon {'f1': 0.234375, 'precision': 0.19230769230769232, 'recall': 0.3}
PrecedentOverruled {'f1': 0.0, 'precision': 0.0, 'r

In [23]:
require_complete(model1_lora_result)
model1_lora_submission = export_experiment("model1", "lora")


/kaggle/working/phase3_rr_v2/direct/model1/lora/test_predictions/submission_model1_lora_v2.zip


### LoRMA

Artifacts: `phase3_rr_v2/direct/model1/lorma/`


In [24]:
model1_lorma_result = train_experiment("model1", "lorma")
print(model1_lorma_result)


model1 lorma {'batch_per_gpu': 1, 'checkpointing': False, 'seconds': 0.192, 'min_free_gib': 12.65}
model1 lorma {'batch_per_gpu': 2, 'checkpointing': False, 'seconds': 0.194, 'min_free_gib': 12.69}
model1 lorma {'batch_per_gpu': 4, 'checkpointing': False, 'seconds': 0.228, 'min_free_gib': 12.39}
model1 lorma {'batch_per_gpu': 8, 'checkpointing': False, 'seconds': 0.349, 'min_free_gib': 11.44}
model1 lorma {'batch_per_gpu': 1, 'checkpointing': True, 'seconds': 0.308, 'min_free_gib': 12.7}
model1 lorma {'batch_per_gpu': 2, 'checkpointing': True, 'seconds': 0.304, 'min_free_gib': 12.7}
model1 lorma {'batch_per_gpu': 4, 'checkpointing': True, 'seconds': 0.321, 'min_free_gib': 12.67}
model1 lorma {'batch_per_gpu': 8, 'checkpointing': True, 'seconds': 0.453, 'min_free_gib': 12.68}
Selected: model1 lorma 8 examples/GPU
model1 lorma {'total': 123081741, 'trainable': 3265165, 'adapted_modules': 64} updates/epoch: 527 effective batch: 32
model1 lorma {'step': 1, 'epoch': 0, 'loss': 2.77380721008

In [25]:
require_complete(model1_lorma_result)
model1_lorma_validation = validate_experiment("model1", "lorma")


Saved F1: 0.38420853613289646 Reloaded F1: 0.38420853613289646
CL documents: 5
IT documents: 5
Total documents: 10
Total sentences: 2142
Macro-F1:        0.384209
Macro-Precision: 0.401163
Macro-Recall:    0.391667

Fact {'f1': 0.6433408577878104, 'precision': 0.6447963800904978, 'recall': 0.6418918918918919}
Issue {'f1': 0.6896551724137931, 'precision': 0.7142857142857143, 'recall': 0.6666666666666666}
ArgumentPetitioner {'f1': 0.4555052790346908, 'precision': 0.5741444866920152, 'recall': 0.3775}
ArgumentRespondent {'f1': 0.30148619957537154, 'precision': 0.35678391959798994, 'recall': 0.2610294117647059}
Statute {'f1': 0.7490909090909091, 'precision': 0.6645161290322581, 'recall': 0.8583333333333333}
Dissent {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
PrecedentReliedUpon {'f1': 0.41509433962264153, 'precision': 0.39855072463768115, 'recall': 0.4330708661417323}
PrecedentNotReliedUpon {'f1': 0.32116788321167883, 'precision': 0.25287356321839083, 'recall': 0.44}
PrecedentOverruled {'

In [26]:
require_complete(model1_lorma_result)
model1_lorma_submission = export_experiment("model1", "lorma")


/kaggle/working/phase3_rr_v2/direct/model1/lorma/test_predictions/submission_model1_lorma_v2.zip


### IA³

Artifacts: `phase3_rr_v2/direct/model1/ia3/`


In [27]:
model1_ia3_result = train_experiment("model1", "ia3")
print(model1_ia3_result)


model1 ia3 {'batch_per_gpu': 1, 'checkpointing': False, 'seconds': 0.137, 'min_free_gib': 12.67}
model1 ia3 {'batch_per_gpu': 2, 'checkpointing': False, 'seconds': 0.159, 'min_free_gib': 12.67}
model1 ia3 {'batch_per_gpu': 4, 'checkpointing': False, 'seconds': 0.223, 'min_free_gib': 12.39}
model1 ia3 {'batch_per_gpu': 8, 'checkpointing': False, 'seconds': 0.342, 'min_free_gib': 11.29}
model1 ia3 {'batch_per_gpu': 1, 'checkpointing': True, 'seconds': 0.181, 'min_free_gib': 12.74}
model1 ia3 {'batch_per_gpu': 2, 'checkpointing': True, 'seconds': 0.19, 'min_free_gib': 12.72}
model1 ia3 {'batch_per_gpu': 4, 'checkpointing': True, 'seconds': 0.256, 'min_free_gib': 12.74}
model1 ia3 {'batch_per_gpu': 8, 'checkpointing': True, 'seconds': 0.417, 'min_free_gib': 12.78}
Selected: model1 ia3 8 examples/GPU
model1 ia3 {'total': 119905293, 'trainable': 88717, 'adapted_modules': 32} updates/epoch: 527 effective batch: 32
model1 ia3 {'step': 1, 'epoch': 0, 'loss': 2.7738072100802666, 'lr': 5.27704485

In [28]:
require_complete(model1_ia3_result)
model1_ia3_validation = validate_experiment("model1", "ia3")


Saved F1: 0.3688568385507925 Reloaded F1: 0.3688568385507925
CL documents: 5
IT documents: 5
Total documents: 10
Total sentences: 2142
Macro-F1:        0.368857
Macro-Precision: 0.387826
Macro-Recall:    0.370494

Fact {'f1': 0.6335797254487856, 'precision': 0.5964214711729622, 'recall': 0.6756756756756757}
Issue {'f1': 0.46153846153846156, 'precision': 0.5454545454545454, 'recall': 0.4}
ArgumentPetitioner {'f1': 0.46776611694152925, 'precision': 0.5842696629213483, 'recall': 0.39}
ArgumentRespondent {'f1': 0.25510204081632654, 'precision': 0.4166666666666667, 'recall': 0.18382352941176472}
Statute {'f1': 0.7509881422924901, 'precision': 0.7142857142857143, 'recall': 0.7916666666666666}
Dissent {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
PrecedentReliedUpon {'f1': 0.4421906693711968, 'precision': 0.4560669456066946, 'recall': 0.42913385826771655}
PrecedentNotReliedUpon {'f1': 0.2857142857142857, 'precision': 0.2463768115942029, 'recall': 0.34}
PrecedentOverruled {'f1': 0.0, 'precision

In [29]:
require_complete(model1_ia3_result)
model1_ia3_submission = export_experiment("model1", "ia3")


/kaggle/working/phase3_rr_v2/direct/model1/ia3/test_predictions/submission_model1_ia3_v2.zip


### Full fine-tuning

Artifacts: `phase3_rr_v2/direct/model1/full/`


In [30]:
model1_full_result = train_experiment("model1", "full")
print(model1_full_result)


model1 full {'batch_per_gpu': 1, 'checkpointing': False, 'seconds': 0.185, 'min_free_gib': 11.64}
model1 full {'batch_per_gpu': 2, 'checkpointing': False, 'seconds': 0.2, 'min_free_gib': 11.66}
model1 full {'batch_per_gpu': 4, 'checkpointing': False, 'seconds': 0.254, 'min_free_gib': 11.6}
model1 full {'batch_per_gpu': 8, 'checkpointing': False, 'seconds': 0.374, 'min_free_gib': 10.83}
model1 full {'batch_per_gpu': 1, 'checkpointing': True, 'seconds': 0.223, 'min_free_gib': 11.73}
model1 full {'batch_per_gpu': 2, 'checkpointing': True, 'seconds': 0.23, 'min_free_gib': 11.75}
model1 full {'batch_per_gpu': 4, 'checkpointing': True, 'seconds': 0.284, 'min_free_gib': 11.69}
model1 full {'batch_per_gpu': 8, 'checkpointing': True, 'seconds': 0.43, 'min_free_gib': 11.75}
Selected: model1 full 8 examples/GPU
model1 full {'total': 119837709, 'trainable': 119837709, 'adapted_modules': 0} updates/epoch: 527 effective batch: 32
model1 full {'step': 1, 'epoch': 0, 'loss': 2.7738072100802666, 'lr': 

In [31]:
require_complete(model1_full_result)
model1_full_validation = validate_experiment("model1", "full")


Saved F1: 0.4025137041650671 Reloaded F1: 0.4025137041650671
CL documents: 5
IT documents: 5
Total documents: 10
Total sentences: 2142
Macro-F1:        0.402514
Macro-Precision: 0.454915
Macro-Recall:    0.386265

Fact {'f1': 0.6519558676028084, 'precision': 0.5877034358047016, 'recall': 0.7319819819819819}
Issue {'f1': 0.64, 'precision': 0.8, 'recall': 0.5333333333333333}
ArgumentPetitioner {'f1': 0.4715447154471545, 'precision': 0.6744186046511628, 'recall': 0.3625}
ArgumentRespondent {'f1': 0.4232558139534884, 'precision': 0.5759493670886076, 'recall': 0.33455882352941174}
Statute {'f1': 0.7634854771784232, 'precision': 0.7603305785123967, 'recall': 0.7666666666666667}
Dissent {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
PrecedentReliedUpon {'f1': 0.42857142857142855, 'precision': 0.35904255319148937, 'recall': 0.531496062992126}
PrecedentNotReliedUpon {'f1': 0.23255813953488372, 'precision': 0.2777777777777778, 'recall': 0.2}
PrecedentOverruled {'f1': 0.0, 'precision': 0.0, 'recall

In [32]:
require_complete(model1_full_result)
model1_full_submission = export_experiment("model1", "full")


/kaggle/working/phase3_rr_v2/direct/model1/full/test_predictions/submission_model1_full_v2.zip


## Model 2: encoder–decoder


### LoRA

Artifacts: `phase3_rr_v2/direct/model2/lora/`


In [33]:
model2_lora_result = train_experiment("model2", "lora")
print(model2_lora_result)


model2 lora {'batch_per_gpu': 1, 'checkpointing': False, 'seconds': 0.204, 'min_free_gib': 12.63}
model2 lora {'batch_per_gpu': 2, 'checkpointing': False, 'seconds': 0.203, 'min_free_gib': 12.7}
model2 lora {'batch_per_gpu': 4, 'checkpointing': False, 'seconds': 0.211, 'min_free_gib': 12.7}
model2 lora {'batch_per_gpu': 8, 'checkpointing': False, 'seconds': 0.248, 'min_free_gib': 12.44}
model2 lora {'batch_per_gpu': 1, 'checkpointing': True, 'seconds': 0.358, 'min_free_gib': 12.76}
model2 lora {'batch_per_gpu': 2, 'checkpointing': True, 'seconds': 0.351, 'min_free_gib': 12.77}
model2 lora {'batch_per_gpu': 4, 'checkpointing': True, 'seconds': 0.36, 'min_free_gib': 12.78}
model2 lora {'batch_per_gpu': 8, 'checkpointing': True, 'seconds': 0.382, 'min_free_gib': 12.76}
Selected: model2 lora 8 examples/GPU
model2 lora {'total': 125937165, 'trainable': 3580557, 'adapted_modules': 116} updates/epoch: 527 effective batch: 32
model2 lora {'step': 1, 'epoch': 0, 'loss': 3.210109466753298, 'lr':

In [34]:
require_complete(model2_lora_result)
model2_lora_validation = validate_experiment("model2", "lora")


Saved F1: 0.1279901318691364 Reloaded F1: 0.1279901318691364
CL documents: 5
IT documents: 5
Total documents: 10
Total sentences: 2142
Macro-F1:        0.127990
Macro-Precision: 0.155274
Macro-Recall:    0.158379

Fact {'f1': 0.45080763582966227, 'precision': 0.3344226579520697, 'recall': 0.6914414414414415}
Issue {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
ArgumentPetitioner {'f1': 0.018957345971563982, 'precision': 0.18181818181818182, 'recall': 0.01}
ArgumentRespondent {'f1': 0.09278350515463918, 'precision': 0.15517241379310345, 'recall': 0.0661764705882353}
Statute {'f1': 0.5702479338842975, 'precision': 0.5655737704918032, 'recall': 0.575}
Dissent {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
PrecedentReliedUpon {'f1': 0.007692307692307693, 'precision': 0.16666666666666666, 'recall': 0.003937007874015748}
PrecedentNotReliedUpon {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
PrecedentOverruled {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
RulingByLowerCourt {'f1': 0.0, 'precision': 0

In [35]:
require_complete(model2_lora_result)
model2_lora_submission = export_experiment("model2", "lora")


/kaggle/working/phase3_rr_v2/direct/model2/lora/test_predictions/submission_model2_lora_v2.zip


### LoRMA

Artifacts: `phase3_rr_v2/direct/model2/lorma/`


In [62]:
model2_lorma_result = train_experiment("model2", "lorma")
print(model2_lorma_result)


Resuming: {'epoch': 5, 'batch': 530, 'step': 2900, 'best_macro_f1': 0.1340569501608442, 'bad_epochs': 1, 'skipped_updates': 0, 'complete': False}
model2 lorma {'total': 125937165, 'trainable': 3580557, 'adapted_modules': 116} updates/epoch: 527 effective batch: 32
model2 lorma {'step': 2920, 'epoch': 5, 'loss': 2.176755128421904, 'lr': 6.517234998594542e-05, 'gradient_norm': 8.525147438049316, 'skipped_updates': 0, 'elapsed_minutes': 0.21798490288335112}
model2 lorma {'step': 2940, 'epoch': 5, 'loss': 2.110973222440621, 'lr': 6.470850271590367e-05, 'gradient_norm': 8.791463851928711, 'skipped_updates': 0, 'elapsed_minutes': 0.3897048731499429}
model2 lorma {'step': 2960, 'epoch': 5, 'loss': 2.1865489336732935, 'lr': 6.424357100909724e-05, 'gradient_norm': 7.012387275695801, 'skipped_updates': 0, 'elapsed_minutes': 0.5632872469832364}
model2 lorma {'step': 2980, 'epoch': 5, 'loss': 2.072290794529843, 'lr': 6.377760679825462e-05, 'gradient_norm': 6.170443058013916, 'skipped_updates': 0, 

In [63]:
require_complete(model2_lorma_result)
model2_lorma_validation = validate_experiment("model2", "lorma")


Saved F1: 0.1340569501608442 Reloaded F1: 0.1340569501608442
CL documents: 5
IT documents: 5
Total documents: 10
Total sentences: 2142
Macro-F1:        0.134057
Macro-Precision: 0.160135
Macro-Recall:    0.149003

Fact {'f1': 0.4498816101026046, 'precision': 0.3462940461725395, 'recall': 0.6418918918918919}
Issue {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
ArgumentPetitioner {'f1': 0.22289156626506024, 'precision': 0.2803030303030303, 'recall': 0.185}
ArgumentRespondent {'f1': 0.11320754716981132, 'precision': 0.21212121212121213, 'recall': 0.07720588235294118}
Statute {'f1': 0.5026178010471204, 'precision': 0.676056338028169, 'recall': 0.4}
Dissent {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
PrecedentReliedUpon {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
PrecedentNotReliedUpon {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
PrecedentOverruled {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
RulingByLowerCourt {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
RatioOfTheDecision {'f1': 0.3211678

In [64]:
require_complete(model2_lorma_result)
model2_lorma_submission = export_experiment("model2", "lorma")


/kaggle/working/phase3_rr_v2/direct/model2/lorma/test_predictions/submission_model2_lorma_v2.zip


### IA³

Artifacts: `phase3_rr_v2/direct/model2/ia3/`


In [65]:
model2_ia3_result = train_experiment("model2", "ia3")
print(model2_ia3_result)


model2 ia3 {'batch_per_gpu': 1, 'checkpointing': False, 'seconds': 0.149, 'min_free_gib': 11.55}
model2 ia3 {'batch_per_gpu': 2, 'checkpointing': False, 'seconds': 0.156, 'min_free_gib': 11.6}
model2 ia3 {'batch_per_gpu': 4, 'checkpointing': False, 'seconds': 0.173, 'min_free_gib': 11.6}
model2 ia3 {'batch_per_gpu': 8, 'checkpointing': False, 'seconds': 0.219, 'min_free_gib': 11.25}
model2 ia3 {'batch_per_gpu': 1, 'checkpointing': True, 'seconds': 0.24, 'min_free_gib': 11.67}
model2 ia3 {'batch_per_gpu': 2, 'checkpointing': True, 'seconds': 0.236, 'min_free_gib': 11.66}
model2 ia3 {'batch_per_gpu': 4, 'checkpointing': True, 'seconds': 0.251, 'min_free_gib': 11.66}
model2 ia3 {'batch_per_gpu': 8, 'checkpointing': True, 'seconds': 0.279, 'min_free_gib': 11.68}
Selected: model2 ia3 8 examples/GPU
model2 ia3 {'total': 122448141, 'trainable': 91533, 'adapted_modules': 58} updates/epoch: 527 effective batch: 32
model2 ia3 {'step': 1, 'epoch': 0, 'loss': 3.210109466753298, 'lr': 5.27704485488

In [66]:
require_complete(model2_ia3_result)
model2_ia3_validation = validate_experiment("model2", "ia3")


Saved F1: 0.09498027667361585 Reloaded F1: 0.09498027667361585
CL documents: 5
IT documents: 5
Total documents: 10
Total sentences: 2142
Macro-F1:        0.094980
Macro-Precision: 0.102336
Macro-Recall:    0.135751

Fact {'f1': 0.4220730797912006, 'precision': 0.31549609810479373, 'recall': 0.6373873873873874}
Issue {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
ArgumentPetitioner {'f1': 0.024154589371980676, 'precision': 0.35714285714285715, 'recall': 0.0125}
ArgumentRespondent {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
Statute {'f1': 0.3525423728813559, 'precision': 0.29714285714285715, 'recall': 0.43333333333333335}
Dissent {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
PrecedentReliedUpon {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
PrecedentNotReliedUpon {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
PrecedentOverruled {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
RulingByLowerCourt {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
RatioOfTheDecision {'f1': 0.3521832498210451, 'precision': 

In [67]:
require_complete(model2_ia3_result)
model2_ia3_submission = export_experiment("model2", "ia3")


/kaggle/working/phase3_rr_v2/direct/model2/ia3/test_predictions/submission_model2_ia3_v2.zip


### Full fine-tuning

Artifacts: `phase3_rr_v2/direct/model2/full/`


In [68]:
model2_full_result = train_experiment("model2", "full")
print(model2_full_result)


model2 full {'batch_per_gpu': 1, 'checkpointing': False, 'seconds': 0.203, 'min_free_gib': 10.56}
model2 full {'batch_per_gpu': 2, 'checkpointing': False, 'seconds': 0.208, 'min_free_gib': 10.63}
model2 full {'batch_per_gpu': 4, 'checkpointing': False, 'seconds': 0.225, 'min_free_gib': 10.58}
model2 full {'batch_per_gpu': 8, 'checkpointing': False, 'seconds': 0.259, 'min_free_gib': 10.44}
model2 full {'batch_per_gpu': 1, 'checkpointing': True, 'seconds': 0.294, 'min_free_gib': 10.74}
model2 full {'batch_per_gpu': 2, 'checkpointing': True, 'seconds': 0.298, 'min_free_gib': 10.71}
model2 full {'batch_per_gpu': 4, 'checkpointing': True, 'seconds': 0.317, 'min_free_gib': 10.68}
model2 full {'batch_per_gpu': 8, 'checkpointing': True, 'seconds': 0.318, 'min_free_gib': 10.67}
Selected: model2 full 8 examples/GPU
model2 full {'total': 122377741, 'trainable': 122377741, 'adapted_modules': 0} updates/epoch: 527 effective batch: 32
model2 full {'step': 1, 'epoch': 0, 'loss': 3.210109466753298, 'l

In [69]:
require_complete(model2_full_result)
model2_full_validation = validate_experiment("model2", "full")


Saved F1: 0.19679925065370057 Reloaded F1: 0.19679925065370057
CL documents: 5
IT documents: 5
Total documents: 10
Total sentences: 2142
Macro-F1:        0.196799
Macro-Precision: 0.211538
Macro-Recall:    0.194033

Fact {'f1': 0.5033860045146726, 'precision': 0.504524886877828, 'recall': 0.5022522522522522}
Issue {'f1': 0.07692307692307693, 'precision': 0.09090909090909091, 'recall': 0.06666666666666667}
ArgumentPetitioner {'f1': 0.2247191011235955, 'precision': 0.31390134529147984, 'recall': 0.175}
ArgumentRespondent {'f1': 0.25968109339407747, 'precision': 0.3413173652694611, 'recall': 0.20955882352941177}
Statute {'f1': 0.6200873362445415, 'precision': 0.6513761467889908, 'recall': 0.5916666666666667}
Dissent {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
PrecedentReliedUpon {'f1': 0.2505399568034557, 'precision': 0.27751196172248804, 'recall': 0.2283464566929134}
PrecedentNotReliedUpon {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
PrecedentOverruled {'f1': 0.0, 'precision': 0.0, 'rec

In [70]:
require_complete(model2_full_result)
model2_full_submission = export_experiment("model2", "full")


/kaggle/working/phase3_rr_v2/direct/model2/full/test_predictions/submission_model2_full_v2.zip


## Model 3: local/global decoder


### LoRA

Artifacts: `phase3_rr_v2/direct/model3/lora/`


In [38]:
model3_lora_result = train_experiment("model3", "lora")
print(model3_lora_result)


model3 lora {'batch_per_gpu': 1, 'checkpointing': False, 'seconds': 0.163, 'min_free_gib': 11.52}
model3 lora {'batch_per_gpu': 2, 'checkpointing': False, 'seconds': 0.177, 'min_free_gib': 11.56}
model3 lora {'batch_per_gpu': 4, 'checkpointing': False, 'seconds': 0.243, 'min_free_gib': 11.3}
model3 lora {'batch_per_gpu': 8, 'checkpointing': False, 'seconds': 0.384, 'min_free_gib': 10.33}
model3 lora {'batch_per_gpu': 1, 'checkpointing': True, 'seconds': 0.298, 'min_free_gib': 11.58}
model3 lora {'batch_per_gpu': 2, 'checkpointing': True, 'seconds': 0.265, 'min_free_gib': 11.58}
model3 lora {'batch_per_gpu': 4, 'checkpointing': True, 'seconds': 0.302, 'min_free_gib': 11.56}
model3 lora {'batch_per_gpu': 8, 'checkpointing': True, 'seconds': 0.498, 'min_free_gib': 11.54}
Selected: model3 lora 8 examples/GPU
model3 lora {'total': 121279501, 'trainable': 2904717, 'adapted_modules': 64} updates/epoch: 527 effective batch: 32
model3 lora {'step': 1, 'epoch': 0, 'loss': 2.6501667371256334, 'lr

In [39]:
require_complete(model3_lora_result)
model3_lora_validation = validate_experiment("model3", "lora")


Saved F1: 0.3840894386803199 Reloaded F1: 0.3840894386803199
CL documents: 5
IT documents: 5
Total documents: 10
Total sentences: 2142
Macro-F1:        0.384089
Macro-Precision: 0.400164
Macro-Recall:    0.402052

Fact {'f1': 0.6387665198237885, 'precision': 0.625, 'recall': 0.6531531531531531}
Issue {'f1': 0.5714285714285714, 'precision': 0.6153846153846154, 'recall': 0.5333333333333333}
ArgumentPetitioner {'f1': 0.3881578947368421, 'precision': 0.5673076923076923, 'recall': 0.295}
ArgumentRespondent {'f1': 0.32923832923832924, 'precision': 0.4962962962962963, 'recall': 0.24632352941176472}
Statute {'f1': 0.7480916030534351, 'precision': 0.6901408450704225, 'recall': 0.8166666666666667}
Dissent {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
PrecedentReliedUpon {'f1': 0.369620253164557, 'precision': 0.5177304964539007, 'recall': 0.2874015748031496}
PrecedentNotReliedUpon {'f1': 0.3356643356643357, 'precision': 0.25806451612903225, 'recall': 0.48}
PrecedentOverruled {'f1': 0.0, 'precision

In [40]:
require_complete(model3_lora_result)
model3_lora_submission = export_experiment("model3", "lora")


/kaggle/working/phase3_rr_v2/direct/model3/lora/test_predictions/submission_model3_lora_v2.zip


### LoRMA

Artifacts: `phase3_rr_v2/direct/model3/lorma/`


In [41]:
model3_lorma_result = train_experiment("model3", "lorma")
print(model3_lorma_result)


model3 lorma {'batch_per_gpu': 1, 'checkpointing': False, 'seconds': 0.204, 'min_free_gib': 11.51}
model3 lorma {'batch_per_gpu': 2, 'checkpointing': False, 'seconds': 0.201, 'min_free_gib': 11.56}
model3 lorma {'batch_per_gpu': 4, 'checkpointing': False, 'seconds': 0.254, 'min_free_gib': 11.16}
model3 lorma {'batch_per_gpu': 8, 'checkpointing': False, 'seconds': 0.385, 'min_free_gib': 10.03}
model3 lorma {'batch_per_gpu': 1, 'checkpointing': True, 'seconds': 0.334, 'min_free_gib': 11.57}
model3 lorma {'batch_per_gpu': 2, 'checkpointing': True, 'seconds': 0.333, 'min_free_gib': 11.58}
model3 lorma {'batch_per_gpu': 4, 'checkpointing': True, 'seconds': 0.349, 'min_free_gib': 11.55}
model3 lorma {'batch_per_gpu': 8, 'checkpointing': True, 'seconds': 0.503, 'min_free_gib': 11.54}
Selected: model3 lorma 8 examples/GPU
model3 lorma {'total': 121639949, 'trainable': 3265165, 'adapted_modules': 64} updates/epoch: 527 effective batch: 32
model3 lorma {'step': 1, 'epoch': 0, 'loss': 2.650166737

In [42]:
require_complete(model3_lorma_result)
model3_lorma_validation = validate_experiment("model3", "lorma")


Saved F1: 0.38867854238520905 Reloaded F1: 0.38867854238520905
CL documents: 5
IT documents: 5
Total documents: 10
Total sentences: 2142
Macro-F1:        0.388679
Macro-Precision: 0.426593
Macro-Recall:    0.387297

Fact {'f1': 0.6361607142857143, 'precision': 0.6305309734513275, 'recall': 0.6418918918918919}
Issue {'f1': 0.6206896551724138, 'precision': 0.6428571428571429, 'recall': 0.6}
ArgumentPetitioner {'f1': 0.44881889763779526, 'precision': 0.4723756906077348, 'recall': 0.4275}
ArgumentRespondent {'f1': 0.2891566265060241, 'precision': 0.4195804195804196, 'recall': 0.22058823529411764}
Statute {'f1': 0.7482014388489209, 'precision': 0.6582278481012658, 'recall': 0.8666666666666667}
Dissent {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
PrecedentReliedUpon {'f1': 0.47297297297297297, 'precision': 0.41420118343195267, 'recall': 0.5511811023622047}
PrecedentNotReliedUpon {'f1': 0.3050847457627119, 'precision': 0.2647058823529412, 'recall': 0.36}
PrecedentOverruled {'f1': 0.0, 'precis

In [43]:
require_complete(model3_lorma_result)
model3_lorma_submission = export_experiment("model3", "lorma")


/kaggle/working/phase3_rr_v2/direct/model3/lorma/test_predictions/submission_model3_lorma_v2.zip


### IA³

Artifacts: `phase3_rr_v2/direct/model3/ia3/`


In [44]:
model3_ia3_result = train_experiment("model3", "ia3")
print(model3_ia3_result)


model3 ia3 {'batch_per_gpu': 1, 'checkpointing': False, 'seconds': 0.146, 'min_free_gib': 11.57}
model3 ia3 {'batch_per_gpu': 2, 'checkpointing': False, 'seconds': 0.178, 'min_free_gib': 11.56}
model3 ia3 {'batch_per_gpu': 4, 'checkpointing': False, 'seconds': 0.248, 'min_free_gib': 11.1}
model3 ia3 {'batch_per_gpu': 8, 'checkpointing': False, 'seconds': 0.39, 'min_free_gib': 9.85}
model3 ia3 {'batch_per_gpu': 1, 'checkpointing': True, 'seconds': 0.218, 'min_free_gib': 11.6}
model3 ia3 {'batch_per_gpu': 2, 'checkpointing': True, 'seconds': 0.244, 'min_free_gib': 11.59}
model3 ia3 {'batch_per_gpu': 4, 'checkpointing': True, 'seconds': 0.297, 'min_free_gib': 11.58}
model3 ia3 {'batch_per_gpu': 8, 'checkpointing': True, 'seconds': 0.487, 'min_free_gib': 11.59}
Selected: model3 ia3 8 examples/GPU
model3 ia3 {'total': 118463501, 'trainable': 88717, 'adapted_modules': 32} updates/epoch: 527 effective batch: 32
model3 ia3 {'step': 1, 'epoch': 0, 'loss': 2.6501667371256334, 'lr': 5.27704485488

In [45]:
require_complete(model3_ia3_result)
model3_ia3_validation = validate_experiment("model3", "ia3")


Saved F1: 0.37570541445598316 Reloaded F1: 0.37570541445598316
CL documents: 5
IT documents: 5
Total documents: 10
Total sentences: 2142
Macro-F1:        0.375705
Macro-Precision: 0.381500
Macro-Recall:    0.386989

Fact {'f1': 0.6176470588235294, 'precision': 0.5787401574803149, 'recall': 0.6621621621621622}
Issue {'f1': 0.6, 'precision': 0.6, 'recall': 0.6}
ArgumentPetitioner {'f1': 0.4357864357864358, 'precision': 0.515358361774744, 'recall': 0.3775}
ArgumentRespondent {'f1': 0.2964824120603015, 'precision': 0.46825396825396826, 'recall': 0.21691176470588236}
Statute {'f1': 0.7565543071161048, 'precision': 0.6870748299319728, 'recall': 0.8416666666666667}
Dissent {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
PrecedentReliedUpon {'f1': 0.351931330472103, 'precision': 0.3867924528301887, 'recall': 0.3228346456692913}
PrecedentNotReliedUpon {'f1': 0.2903225806451613, 'precision': 0.24324324324324326, 'recall': 0.36}
PrecedentOverruled {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
RulingB

In [46]:
require_complete(model3_ia3_result)
model3_ia3_submission = export_experiment("model3", "ia3")


/kaggle/working/phase3_rr_v2/direct/model3/ia3/test_predictions/submission_model3_ia3_v2.zip


### Full fine-tuning

Artifacts: `phase3_rr_v2/direct/model3/full/`


In [47]:
model3_full_result = train_experiment("model3", "full")
print(model3_full_result)


model3 full {'batch_per_gpu': 1, 'checkpointing': False, 'seconds': 0.197, 'min_free_gib': 10.5}
model3 full {'batch_per_gpu': 2, 'checkpointing': False, 'seconds': 0.216, 'min_free_gib': 10.5}
model3 full {'batch_per_gpu': 4, 'checkpointing': False, 'seconds': 0.275, 'min_free_gib': 10.36}
model3 full {'batch_per_gpu': 8, 'checkpointing': False, 'seconds': 0.402, 'min_free_gib': 9.38}
model3 full {'batch_per_gpu': 1, 'checkpointing': True, 'seconds': 0.27, 'min_free_gib': 10.59}
model3 full {'batch_per_gpu': 2, 'checkpointing': True, 'seconds': 0.278, 'min_free_gib': 10.62}
model3 full {'batch_per_gpu': 4, 'checkpointing': True, 'seconds': 0.313, 'min_free_gib': 10.51}
model3 full {'batch_per_gpu': 8, 'checkpointing': True, 'seconds': 0.477, 'min_free_gib': 10.6}
Selected: model3 full 8 examples/GPU
model3 full {'total': 118395917, 'trainable': 118395917, 'adapted_modules': 0} updates/epoch: 527 effective batch: 32
model3 full {'step': 1, 'epoch': 0, 'loss': 2.6501667371256334, 'lr': 

In [48]:
require_complete(model3_full_result)
model3_full_validation = validate_experiment("model3", "full")


Saved F1: 0.3906423893368165 Reloaded F1: 0.3906423893368165
CL documents: 5
IT documents: 5
Total documents: 10
Total sentences: 2142
Macro-F1:        0.390642
Macro-Precision: 0.417372
Macro-Recall:    0.381204

Fact {'f1': 0.6508108108108108, 'precision': 0.6257796257796258, 'recall': 0.6779279279279279}
Issue {'f1': 0.7142857142857143, 'precision': 0.7692307692307693, 'recall': 0.6666666666666666}
ArgumentPetitioner {'f1': 0.49326145552560646, 'precision': 0.5350877192982456, 'recall': 0.4575}
ArgumentRespondent {'f1': 0.26595744680851063, 'precision': 0.4807692307692308, 'recall': 0.18382352941176472}
Statute {'f1': 0.772093023255814, 'precision': 0.8736842105263158, 'recall': 0.6916666666666667}
Dissent {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
PrecedentReliedUpon {'f1': 0.36470588235294116, 'precision': 0.36328125, 'recall': 0.3661417322834646}
PrecedentNotReliedUpon {'f1': 0.3711340206185567, 'precision': 0.3829787234042553, 'recall': 0.36}
PrecedentOverruled {'f1': 0.0, 'pr

In [49]:
require_complete(model3_full_result)
model3_full_submission = export_experiment("model3", "full")


/kaggle/working/phase3_rr_v2/direct/model3/full/test_predictions/submission_model3_full_v2.zip


## Model 4: gated convolution


### LoRA

Artifacts: `phase3_rr_v2/direct/model4/lora/`


In [50]:
model4_lora_result = train_experiment("model4", "lora")
print(model4_lora_result)


model4 lora {'batch_per_gpu': 1, 'checkpointing': False, 'seconds': 0.17, 'min_free_gib': 11.52}
model4 lora {'batch_per_gpu': 2, 'checkpointing': False, 'seconds': 0.169, 'min_free_gib': 11.56}
model4 lora {'batch_per_gpu': 4, 'checkpointing': False, 'seconds': 0.206, 'min_free_gib': 11.51}
model4 lora {'batch_per_gpu': 8, 'checkpointing': False, 'seconds': 0.314, 'min_free_gib': 10.78}
model4 lora {'batch_per_gpu': 1, 'checkpointing': True, 'seconds': 0.251, 'min_free_gib': 11.56}
model4 lora {'batch_per_gpu': 2, 'checkpointing': True, 'seconds': 0.248, 'min_free_gib': 11.58}
model4 lora {'batch_per_gpu': 4, 'checkpointing': True, 'seconds': 0.264, 'min_free_gib': 11.56}
model4 lora {'batch_per_gpu': 8, 'checkpointing': True, 'seconds': 0.407, 'min_free_gib': 11.57}
Selected: model4 lora 8 examples/GPU
model4 lora {'total': 121384973, 'trainable': 1985293, 'adapted_modules': 48} updates/epoch: 527 effective batch: 32
model4 lora {'step': 1, 'epoch': 0, 'loss': 2.7410128169116654, 'lr

In [51]:
require_complete(model4_lora_result)
model4_lora_validation = validate_experiment("model4", "lora")


Saved F1: 0.3415094948155011 Reloaded F1: 0.3415094948155011
CL documents: 5
IT documents: 5
Total documents: 10
Total sentences: 2142
Macro-F1:        0.341509
Macro-Precision: 0.343843
Macro-Recall:    0.357714

Fact {'f1': 0.5850945494994438, 'precision': 0.578021978021978, 'recall': 0.5923423423423423}
Issue {'f1': 0.3333333333333333, 'precision': 0.2857142857142857, 'recall': 0.4}
ArgumentPetitioner {'f1': 0.4539385847797063, 'precision': 0.4871060171919771, 'recall': 0.425}
ArgumentRespondent {'f1': 0.19786096256684493, 'precision': 0.3627450980392157, 'recall': 0.13602941176470587}
Statute {'f1': 0.7434944237918215, 'precision': 0.6711409395973155, 'recall': 0.8333333333333334}
Dissent {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
PrecedentReliedUpon {'f1': 0.47377938517179025, 'precision': 0.43812709030100333, 'recall': 0.515748031496063}
PrecedentNotReliedUpon {'f1': 0.2204724409448819, 'precision': 0.18181818181818182, 'recall': 0.28}
PrecedentOverruled {'f1': 0.0, 'precision'

In [52]:
require_complete(model4_lora_result)
model4_lora_submission = export_experiment("model4", "lora")


/kaggle/working/phase3_rr_v2/direct/model4/lora/test_predictions/submission_model4_lora_v2.zip


### LoRMA

Artifacts: `phase3_rr_v2/direct/model4/lorma/`


In [53]:
model4_lorma_result = train_experiment("model4", "lorma")
print(model4_lorma_result)


model4 lorma {'batch_per_gpu': 1, 'checkpointing': False, 'seconds': 0.183, 'min_free_gib': 11.51}
model4 lorma {'batch_per_gpu': 2, 'checkpointing': False, 'seconds': 0.192, 'min_free_gib': 11.55}
model4 lorma {'batch_per_gpu': 4, 'checkpointing': False, 'seconds': 0.211, 'min_free_gib': 11.5}
model4 lorma {'batch_per_gpu': 8, 'checkpointing': False, 'seconds': 0.315, 'min_free_gib': 10.76}
model4 lorma {'batch_per_gpu': 1, 'checkpointing': True, 'seconds': 0.266, 'min_free_gib': 11.56}
model4 lorma {'batch_per_gpu': 2, 'checkpointing': True, 'seconds': 0.281, 'min_free_gib': 11.58}
model4 lorma {'batch_per_gpu': 4, 'checkpointing': True, 'seconds': 0.275, 'min_free_gib': 11.56}
model4 lorma {'batch_per_gpu': 8, 'checkpointing': True, 'seconds': 0.403, 'min_free_gib': 11.57}
Selected: model4 lorma 8 examples/GPU
model4 lorma {'total': 121384973, 'trainable': 1985293, 'adapted_modules': 48} updates/epoch: 527 effective batch: 32
model4 lorma {'step': 1, 'epoch': 0, 'loss': 2.7410128169

In [54]:
require_complete(model4_lorma_result)
model4_lorma_validation = validate_experiment("model4", "lorma")


Saved F1: 0.3553505919509919 Reloaded F1: 0.3553505919509919
CL documents: 5
IT documents: 5
Total documents: 10
Total sentences: 2142
Macro-F1:        0.355351
Macro-Precision: 0.356002
Macro-Recall:    0.369620

Fact {'f1': 0.5928411633109619, 'precision': 0.5888888888888889, 'recall': 0.5968468468468469}
Issue {'f1': 0.35294117647058826, 'precision': 0.3157894736842105, 'recall': 0.4}
ArgumentPetitioner {'f1': 0.4696969696969697, 'precision': 0.4744897959183674, 'recall': 0.465}
ArgumentRespondent {'f1': 0.2383177570093458, 'precision': 0.3269230769230769, 'recall': 0.1875}
Statute {'f1': 0.7298245614035088, 'precision': 0.6303030303030303, 'recall': 0.8666666666666667}
Dissent {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
PrecedentReliedUpon {'f1': 0.4753521126760563, 'precision': 0.4299363057324841, 'recall': 0.531496062992126}
PrecedentNotReliedUpon {'f1': 0.27906976744186046, 'precision': 0.22784810126582278, 'recall': 0.36}
PrecedentOverruled {'f1': 0.0, 'precision': 0.0, 'recal

In [55]:
require_complete(model4_lorma_result)
model4_lorma_submission = export_experiment("model4", "lorma")


/kaggle/working/phase3_rr_v2/direct/model4/lorma/test_predictions/submission_model4_lorma_v2.zip


### IA³

Artifacts: `phase3_rr_v2/direct/model4/ia3/`


In [56]:
model4_ia3_result = train_experiment("model4", "ia3")
print(model4_ia3_result)


model4 ia3 {'batch_per_gpu': 1, 'checkpointing': False, 'seconds': 0.149, 'min_free_gib': 11.56}
model4 ia3 {'batch_per_gpu': 2, 'checkpointing': False, 'seconds': 0.169, 'min_free_gib': 11.55}
model4 ia3 {'batch_per_gpu': 4, 'checkpointing': False, 'seconds': 0.215, 'min_free_gib': 11.23}
model4 ia3 {'batch_per_gpu': 8, 'checkpointing': False, 'seconds': 0.321, 'min_free_gib': 10.18}
model4 ia3 {'batch_per_gpu': 1, 'checkpointing': True, 'seconds': 0.201, 'min_free_gib': 11.57}
model4 ia3 {'batch_per_gpu': 2, 'checkpointing': True, 'seconds': 0.241, 'min_free_gib': 11.56}
model4 ia3 {'batch_per_gpu': 4, 'checkpointing': True, 'seconds': 0.258, 'min_free_gib': 11.57}
model4 ia3 {'batch_per_gpu': 8, 'checkpointing': True, 'seconds': 0.408, 'min_free_gib': 11.6}
Selected: model4 ia3 8 examples/GPU
model4 ia3 {'total': 119480333, 'trainable': 80653, 'adapted_modules': 32} updates/epoch: 527 effective batch: 32
model4 ia3 {'step': 1, 'epoch': 0, 'loss': 2.741075118716266, 'lr': 5.277044854

In [57]:
require_complete(model4_ia3_result)
model4_ia3_validation = validate_experiment("model4", "ia3")


Saved F1: 0.31426387059062016 Reloaded F1: 0.31426387059062016
CL documents: 5
IT documents: 5
Total documents: 10
Total sentences: 2142
Macro-F1:        0.314264
Macro-Precision: 0.338323
Macro-Recall:    0.316774

Fact {'f1': 0.5739130434782609, 'precision': 0.5546218487394958, 'recall': 0.5945945945945946}
Issue {'f1': 0.2608695652173913, 'precision': 0.375, 'recall': 0.2}
ArgumentPetitioner {'f1': 0.38971583220568334, 'precision': 0.4247787610619469, 'recall': 0.36}
ArgumentRespondent {'f1': 0.09523809523809523, 'precision': 0.25, 'recall': 0.058823529411764705}
Statute {'f1': 0.782608695652174, 'precision': 0.7443609022556391, 'recall': 0.825}
Dissent {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
PrecedentReliedUpon {'f1': 0.4046242774566474, 'precision': 0.39622641509433965, 'recall': 0.41338582677165353}
PrecedentNotReliedUpon {'f1': 0.2619047619047619, 'precision': 0.3235294117647059, 'recall': 0.22}
PrecedentOverruled {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
RulingByLowerCo

In [58]:
require_complete(model4_ia3_result)
model4_ia3_submission = export_experiment("model4", "ia3")


/kaggle/working/phase3_rr_v2/direct/model4/ia3/test_predictions/submission_model4_ia3_v2.zip


### Full fine-tuning

Artifacts: `phase3_rr_v2/direct/model4/full/`


In [59]:
model4_full_result = train_experiment("model4", "full")
print(model4_full_result)


model4 full {'batch_per_gpu': 1, 'checkpointing': False, 'seconds': 0.2, 'min_free_gib': 10.52}
model4 full {'batch_per_gpu': 2, 'checkpointing': False, 'seconds': 0.224, 'min_free_gib': 10.55}
model4 full {'batch_per_gpu': 4, 'checkpointing': False, 'seconds': 0.269, 'min_free_gib': 10.52}
model4 full {'batch_per_gpu': 8, 'checkpointing': False, 'seconds': 0.379, 'min_free_gib': 9.81}
model4 full {'batch_per_gpu': 1, 'checkpointing': True, 'seconds': 0.239, 'min_free_gib': 10.61}
model4 full {'batch_per_gpu': 2, 'checkpointing': True, 'seconds': 0.255, 'min_free_gib': 10.6}
model4 full {'batch_per_gpu': 4, 'checkpointing': True, 'seconds': 0.311, 'min_free_gib': 10.52}
model4 full {'batch_per_gpu': 8, 'checkpointing': True, 'seconds': 0.448, 'min_free_gib': 10.62}
Selected: model4 full 8 examples/GPU
model4 full {'total': 119418893, 'trainable': 119418893, 'adapted_modules': 0} updates/epoch: 527 effective batch: 32
model4 full {'step': 1, 'epoch': 0, 'loss': 2.7410128169116654, 'lr':

In [60]:
require_complete(model4_full_result)
model4_full_validation = validate_experiment("model4", "full")


Saved F1: 0.35695357046430687 Reloaded F1: 0.35695357046430687
CL documents: 5
IT documents: 5
Total documents: 10
Total sentences: 2142
Macro-F1:        0.356954
Macro-Precision: 0.371907
Macro-Recall:    0.361782

Fact {'f1': 0.5552995391705069, 'precision': 0.5683962264150944, 'recall': 0.5427927927927928}
Issue {'f1': 0.5, 'precision': 0.6666666666666666, 'recall': 0.4}
ArgumentPetitioner {'f1': 0.4011379800853485, 'precision': 0.46534653465346537, 'recall': 0.3525}
ArgumentRespondent {'f1': 0.3388235294117647, 'precision': 0.47058823529411764, 'recall': 0.2647058823529412}
Statute {'f1': 0.7391304347826086, 'precision': 0.6538461538461539, 'recall': 0.85}
Dissent {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
PrecedentReliedUpon {'f1': 0.4801670146137787, 'precision': 0.5111111111111111, 'recall': 0.452755905511811}
PrecedentNotReliedUpon {'f1': 0.24242424242424243, 'precision': 0.1951219512195122, 'recall': 0.32}
PrecedentOverruled {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
Rulin

In [61]:
require_complete(model4_full_result)
model4_full_submission = export_experiment("model4", "full")


/kaggle/working/phase3_rr_v2/direct/model4/full/test_predictions/submission_model4_full_v2.zip


## Compare completed runs


In [71]:
def collect_summaries():
    rows = []
    for model_name in MODEL_NAMES:
        for method in DIRECT_METHODS:
            path = experiment_folder(model_name, method) / "summary.json"
            if path.is_file():
                rows.append(read_json(path))
    return rows

rows = collect_summaries()
print(f"{'Model':<10} {'Method':<10} {'Overall':>10} {'CL':>10} {'IT':>10} {'Trainable':>14}")
print("-" * 70)
for row in sorted(rows, key=lambda item: item["best_macro_f1"], reverse=True):
    print(f"{row['model_name']:<10} {row['method']:<10} {row['best_macro_f1']:>10.6f} "
          f"{row['domains']['CL']['macro_f1']:>10.6f} {row['domains']['IT']['macro_f1']:>10.6f} "
          f"{row['parameters']['trainable']:>14,d}")


Model      Method        Overall         CL         IT      Trainable
----------------------------------------------------------------------
model1     full         0.402514   0.324982   0.419667    119,837,709
model3     full         0.390642   0.304827   0.422457    118,395,917
model3     lorma        0.388679   0.262012   0.429711      3,265,165
model1     lorma        0.384209   0.301351   0.409055      3,265,165
model3     lora         0.384089   0.274164   0.412258      2,904,717
model1     lora         0.377642   0.286848   0.399343      2,904,717
model3     ia3          0.375705   0.290079   0.410433         88,717
model1     ia3          0.368857   0.276543   0.391527         88,717
model4     full         0.356954   0.257657   0.402153    119,418,893
model4     lorma        0.355351   0.269472   0.391719      1,985,293
model4     lora         0.341509   0.266734   0.372583      1,985,293
model4     ia3          0.314264   0.235449   0.344174         80,653
model2     full    

## Knowledge distillation

Run this section after direct experiments have produced a teacher. The default student is the smallest Phase 1 model. The teacher is the best completed direct experiment on a larger Phase 1 model. Its logits are cached only for training sentences, in exactly the same order as the student records. Development labels are used for checkpoint selection. You can set the teacher explicitly before running this section.


In [72]:
KD_STUDENT_MODEL = None
KD_TEACHER_SPEC = None

def choose_kd_pair():
    student = KD_STUDENT_MODEL or min(MODEL_NAMES, key=lambda name: base_report[name]["parameters"])
    student_size = base_report[student]["parameters"]
    candidates = []
    for row in collect_summaries():
        if base_report[row["model_name"]]["parameters"] > student_size:
            candidates.append(row)
    if KD_TEACHER_SPEC is not None:
        teacher_model, teacher_method = KD_TEACHER_SPEC
        selected = [row for row in candidates if row["model_name"] == teacher_model and row["method"] == teacher_method]
        if len(selected) != 1:
            raise ValueError("Requested teacher is unavailable or is not larger than the student")
        teacher = selected[0]
    else:
        if not candidates:
            raise ValueError("Complete at least one larger direct model before distillation")
        teacher = max(candidates, key=lambda row: row["best_macro_f1"])
    return student, teacher

KD_STUDENT_MODEL, KD_TEACHER = choose_kd_pair()
print("Student:", KD_STUDENT_MODEL, base_report[KD_STUDENT_MODEL]["parameters"])
print("Teacher:", KD_TEACHER["model_name"], KD_TEACHER["method"], KD_TEACHER["best_macro_f1"], base_report[KD_TEACHER["model_name"]]["parameters"])


Student: model3 118374784
Teacher: model1 full 0.4025137041650671 119816576


In [73]:
def cache_teacher_logits(teacher_summary):
    teacher_model = teacher_summary["model_name"]
    teacher_method = teacher_summary["method"]
    teacher_dir = Path(teacher_summary["run_dir"])
    cache_dir = OUTPUT_ROOT / "distillation" / "teacher_cache"
    cache_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_hash = digest_file(teacher_dir / "best.pt")
    cache_path = cache_dir / f"{teacher_model}__{teacher_method}__{checkpoint_hash[:12]}.pt"
    signature = {
        "teacher_checkpoint": checkpoint_hash,
        "teacher_model": teacher_model,
        "teacher_method": teacher_method,
        "records": len(TRAIN_RECORDS),
        "data": DATA_FILES,
        "input_signature": encoded_cache_signature(teacher_model),
        "student_model": KD_STUDENT_MODEL,
        "student_input_signature": encoded_cache_signature(KD_STUDENT_MODEL),
    }
    if cache_path.is_file():
        payload = torch.load(cache_path, map_location="cpu", weights_only=False)
        if payload["signature"] != signature:
            raise ValueError("Teacher cache signature mismatch")
        return payload["logits"].float(), cache_path
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    teacher, config = load_best_model(teacher_model, teacher_method, teacher_dir, device)
    batch = config["inference_batch_size"]
    wrapped = teacher
    logits = infer_logits(wrapped, ENCODED[teacher_model]["train"], batch, device).float()
    if not torch.isfinite(logits).all():
        raise FloatingPointError("Nonfinite teacher logits")
    assert [(e["document_id"], e["sentence_index"]) for e in ENCODED[teacher_model]["train"]] == [(e["document_id"], e["sentence_index"]) for e in ENCODED[KD_STUDENT_MODEL]["train"]]
    torch.save({"signature": signature, "logits": logits}, cache_path)
    del wrapped, teacher
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return logits.float(), cache_path

TEACHER_LOGITS, TEACHER_CACHE = cache_teacher_logits(KD_TEACHER)
print("Teacher cache:", TEACHER_CACHE, TEACHER_LOGITS.shape)


Teacher cache: /kaggle/working/phase3_rr_v2/distillation/teacher_cache/model1__full__5753bdc4ce9c.pt torch.Size([16845, 13])


### Distillation training


In [74]:
teacher_reference = {
    "model_name": KD_TEACHER["model_name"],
    "method": KD_TEACHER["method"],
    "checkpoint": digest_file(Path(KD_TEACHER["run_dir"]) / "best.pt"),
    "cache": digest_file(TEACHER_CACHE),
}
KD_RESULT = train_experiment(
    KD_STUDENT_MODEL,
    "kd",
    teacher_logits=TEACHER_LOGITS,
    teacher_reference=teacher_reference,
)
print(KD_RESULT)


model3 kd {'total': 118395917, 'trainable': 118395917, 'adapted_modules': 0} updates/epoch: 527 effective batch: 32
model3 kd {'step': 1, 'epoch': 0, 'loss': 6.259304603098492, 'lr': 1.0554089709762534e-07, 'gradient_norm': 27.597206115722656, 'skipped_updates': 0, 'elapsed_minutes': 0.03230396318334291}
model3 kd {'step': 20, 'epoch': 0, 'loss': 6.372712408416901, 'lr': 1.108179419525066e-06, 'gradient_norm': 27.819934844970703, 'skipped_updates': 0, 'elapsed_minutes': 0.22782864368333927}
model3 kd {'step': 40, 'epoch': 0, 'loss': 5.979423282498847, 'lr': 2.1635883905013194e-06, 'gradient_norm': 21.8748722076416, 'skipped_updates': 0, 'elapsed_minutes': 0.4338477476667322}
model3 kd {'step': 60, 'epoch': 0, 'loss': 5.507887223330786, 'lr': 3.2189973614775725e-06, 'gradient_norm': 24.208391189575195, 'skipped_updates': 0, 'elapsed_minutes': 0.6393125493667564}
model3 kd {'step': 80, 'epoch': 0, 'loss': 5.42191871226676, 'lr': 4.274406332453826e-06, 'gradient_norm': 22.959815979003906,

In [75]:
require_complete(KD_RESULT)
KD_VALIDATION = validate_experiment(KD_STUDENT_MODEL, "kd")


Saved F1: 0.4169236959608398 Reloaded F1: 0.4169236959608398
CL documents: 5
IT documents: 5
Total documents: 10
Total sentences: 2142
Macro-F1:        0.416924
Macro-Precision: 0.442690
Macro-Recall:    0.411279

Fact {'f1': 0.6616541353383458, 'precision': 0.6324435318275154, 'recall': 0.6936936936936937}
Issue {'f1': 0.7142857142857143, 'precision': 0.7692307692307693, 'recall': 0.6666666666666666}
ArgumentPetitioner {'f1': 0.4947839046199702, 'precision': 0.6125461254612546, 'recall': 0.415}
ArgumentRespondent {'f1': 0.4085510688836104, 'precision': 0.5771812080536913, 'recall': 0.3161764705882353}
Statute {'f1': 0.7786885245901639, 'precision': 0.7661290322580645, 'recall': 0.7916666666666666}
Dissent {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
PrecedentReliedUpon {'f1': 0.3853503184713376, 'precision': 0.3235294117647059, 'recall': 0.4763779527559055}
PrecedentNotReliedUpon {'f1': 0.3119266055045872, 'precision': 0.288135593220339, 'recall': 0.34}
PrecedentOverruled {'f1': 0.0, 

In [76]:
require_complete(KD_RESULT)
KD_SUBMISSION = export_experiment(KD_STUDENT_MODEL, "kd")


/kaggle/working/phase3_rr_v2/distillation/model3/test_predictions/submission_model3_kd_v2.zip


## Four-architecture selection

This optional table chooses KD for its student architecture and assigns LoRA, LoRMA and IA³ once each to the remaining three architectures. All individual submissions have already been exported in their experiment sections. The table records an assignment; it does not establish the current course minimum or submit files to Codabench.


In [77]:
def choose_required_assignment():
    kd_path = OUTPUT_ROOT / "distillation" / KD_STUDENT_MODEL / "summary.json"
    if not kd_path.is_file():
        raise FileNotFoundError("Complete distillation first")
    kd = read_json(kd_path)
    remaining_models = [name for name in MODEL_NAMES if name != KD_STUDENT_MODEL]
    techniques = ("lora", "lorma", "ia3")
    best = None
    for permutation in itertools.permutations(techniques):
        experiments = []
        valid = True
        for model_name, method in zip(remaining_models, permutation):
            path = experiment_folder(model_name, method) / "summary.json"
            if not path.is_file():
                valid = False
                break
            experiments.append(read_json(path))
        if not valid:
            continue
        score = sum(item["best_macro_f1"] for item in experiments) + kd["best_macro_f1"]
        candidate = {"score_sum": score, "experiments": experiments + [kd]}
        if best is None or candidate["score_sum"] > best["score_sum"]:
            best = candidate
    if best is None:
        raise ValueError("Complete LoRA, LoRMA and IA3 for all non-student models")
    return best

ASSIGNMENT = choose_required_assignment()
write_json(OUTPUT_ROOT / "required_assignment.json", ASSIGNMENT)
for item in ASSIGNMENT["experiments"]:
    print(item["model_name"], item["method"], item["best_macro_f1"], item["domains"])


model1 ia3 0.3688568385507925 {'CL': {'macro_f1': 0.276542699604326, 'macro_precision': 0.283729923996435, 'macro_recall': 0.2979870549759394}, 'IT': {'macro_f1': 0.39152654964449246, 'macro_precision': 0.42709051884866717, 'macro_recall': 0.3741450412847003}}
model2 lora 0.1279901318691364 {'CL': {'macro_f1': 0.1229274747211904, 'macro_precision': 0.1567937637456308, 'macro_recall': 0.1625484516570415}, 'IT': {'macro_f1': 0.12617171655011064, 'macro_precision': 0.13071303458883288, 'macro_recall': 0.15327205011097556}}
model4 lorma 0.3553505919509919 {'CL': {'macro_f1': 0.2694719499732694, 'macro_precision': 0.2633641617966431, 'macro_recall': 0.2970071847561116}, 'IT': {'macro_f1': 0.39171912322621577, 'macro_precision': 0.431519575028173, 'macro_recall': 0.3836909141912378}}
model3 kd 0.4169236959608398 {'CL': {'macro_f1': 0.32284666205085305, 'macro_precision': 0.32525570586990626, 'macro_recall': 0.34845524345697704}, 'IT': {'macro_f1': 0.45362769552104215, 'macro_precision': 0.51

## Method references

[LoRA](https://arxiv.org/abs/2106.09685), [LoRMA](https://arxiv.org/abs/2506.07621), [IA³](https://arxiv.org/abs/2205.05638), and [knowledge distillation](https://arxiv.org/abs/1503.02531).